# Lab 1 - The AI Lifecycle, End to End
### Deloitte AI Solution Architect Programme | Day 1 | Hands-on lab (~75-90 min)

> *"A model is not a product. A model that is monitored, governed, and wired into a decision is a product."*

In this lab you will walk **one realistic business problem (telecom customer churn)** through the
complete AI lifecycle - from "should we even do this?" to "the model has drifted, do we retrain?" -
and then zoom out to see the wider AI landscape and the situations where you should say **no** to AI.

## Learning outcomes
By the end of this lab you can:

1. **Place** a use case on the AI / ML / DL / GenAI / Agentic map and say what that choice means for architecture (data, cost, latency, risk).
2. **Distinguish** supervised, unsupervised and reinforcement learning by the *feedback signal* each one needs.
3. **Run** the CRISP-DM phases on one case and explain which artefact each phase produces.
4. **Convert** a business cost matrix into a decision threshold (this is where model quality becomes money).
5. **Explain** why the *pipeline* (not the model) is the deployable artefact, and how drift monitoring closes the MLOps loop.
6. **Challenge** a proposed AI use case with a structured "Should we use AI?" checklist.
7. **Map** the four classic capabilities (NLP, Vision, Speech, Recommenders) to inputs, model families and architecture concerns.

## How to use this notebook
* Run cells top to bottom with **Shift+Enter**. The instructor will narrate; you follow along.
* Cells marked **Your turn** contain a `TODO`. Try it yourself first - the instructor will reveal the solution.
* Everything runs **offline** on a laptop in well under 3 minutes in total. Data is synthetic, generated in code, seeded for reproducibility.
* Three kinds of boxes: **Why** (the reasoning), **Architect lens** (cost / latency / risk / governance), **Takeaway** (what to remember).

## 0. Environment check
**Why:** 90% of "it doesn't work on my machine" in a workshop is a version issue. We print versions once, fix the random seed, and set a consistent chart style so every plot looks the same on the projector.

**Architect lens:** Pinning versions and seeds is the smallest possible instance of *reproducibility*, which later becomes a governance requirement (audit: "can you reproduce the model that made this decision?").

In [ ]:
import os
# Keep BLAS/OpenMP to one thread: tiny data, so threading only adds overhead (and nondeterminism) on shared laptops/servers.
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")
import sys, platform, warnings, json, time, hashlib, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches
from matplotlib.colors import ListedColormap

warnings.filterwarnings("ignore")          # keep the projector output clean

# ---- reproducibility: ONE seed used everywhere ----
SEED = 42
rng = np.random.default_rng(SEED)

# ---- consistent palette ----
INK, TEAL, LIME, CORAL, GOLD = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.labelcolor": INK,
    "text.color": INK, "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.prop_cycle": matplotlib.cycler(color=[TEAL, CORAL, GOLD, LIME, INK]),
    "axes.grid": True, "grid.alpha": 0.25,
})

# ---- where artefacts (saved models etc.) go: relative to the labs folder ----
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

print(f"Python      : {platform.python_version()}")
print(f"numpy       : {np.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"scipy       : {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")
print(f"Seed        : {SEED}   |   artefacts dir: {ARTIFACTS.resolve().name}/")
LAB_START = time.time()

---
# Part A - The AI landscape through an *architect's* lens

## A.1 AI vs ML vs DL vs GenAI vs Agentic AI
These words are used interchangeably in meetings, and that is dangerous: each one implies a **different architecture, cost profile and risk profile**.

Think of Russian dolls:

* **AI** - any technique that makes a machine behave "intelligently". Includes plain **rules engines** and search. (A tax calculator is, historically, "AI" - and often the right answer.)
* **Machine Learning (ML)** - the part of AI where behaviour is **learned from data** instead of hand-coded.
* **Deep Learning (DL)** - ML using many-layered neural networks; shines on unstructured data (images, audio, text).
* **Generative AI (GenAI)** - DL models (LLMs, diffusion) trained to **generate** content: text, code, images. Output is open-ended, probabilistic.
* **Agentic AI** - GenAI wrapped in a **loop with tools and memory**: it plans, calls APIs, observes results, and acts. The *system* makes decisions, not just predictions.

Run the next cell to draw the picture, then the table that matters most to you: what each layer means for **architecture**.

In [ ]:
# Nested "Russian doll" diagram: each layer is a subset of the one before it.
fig, ax = plt.subplots(figsize=(8.5, 5.2))
layers = [  # (label, width, height, colour)
    ("AI  (rules, search, optimisation ...)", 8.0, 5.0, "#E8EEF0"),
    ("Machine Learning  (learn from data)", 6.6, 4.0, "#CFE6E8"),
    ("Deep Learning  (neural networks)", 5.2, 3.0, "#9ACDD2"),
    ("Generative AI  (LLMs, diffusion)", 3.8, 2.0, TEAL),
]
for label, w, h, col in layers:
    ax.add_patch(matplotlib.patches.FancyBboxPatch((4 - w/2, 2.6 - h/2), w, h,
                 boxstyle="round,pad=0.02,rounding_size=0.4", fc=col, ec=INK, lw=1))
    ax.text(4, 2.6 + h/2 - 0.28, label, ha="center", va="center", fontsize=9.5,
            color="white" if col == TEAL else INK, fontweight="bold")
ax.add_patch(matplotlib.patches.FancyBboxPatch((2.55, 1.95), 2.9, 0.8,
             boxstyle="round,pad=0.02,rounding_size=0.3", fc=GOLD, ec=INK, lw=1))
ax.text(4, 2.35, "Agentic AI\n(GenAI + tools + loop)", ha="center", va="center", fontsize=9, fontweight="bold")
ax.set_xlim(-0.3, 8.3); ax.set_ylim(-0.1, 5.3); ax.axis("off"); ax.grid(False)
ax.set_title("The AI nesting dolls", loc="center")
ax.text(4, -0.05, "Each inner layer: more capable, more costly, harder to govern", ha="center", fontsize=9)
plt.show()

In [ ]:
# The architect's comparison table. Same word "AI", very different engineering consequences.
landscape = pd.DataFrame({
    "Paradigm":        ["Rules / classic AI", "Classic ML", "Deep Learning", "Generative AI", "Agentic AI"],
    "Typical input":   ["Structured fields", "Tabular features", "Images / audio / text", "Prompt + context", "Goal + tools + memory"],
    "Typical output":  ["Deterministic decision", "Score / class / number", "Class / embedding", "Open-ended content", "Actions with side effects"],
    "Deterministic?":  ["Yes", "Mostly", "Mostly", "No", "No"],
    "Data need":       ["None (expert knowledge)", "100s - 100,000s labelled rows", "10,000s - millions", "Pre-trained; your docs for RAG", "Pre-trained + tool APIs"],
    "Unit cost / latency": ["~free / ms", "~free / ms", "cheap-moderate / 10s of ms", "per-token $ / seconds", "many LLM calls $$ / 10s of s"],
    "Main governance risk": ["Stale rules", "Bias, drift", "Opacity", "Hallucination, leakage", "Autonomy, blast radius"],
})
pd.set_option("display.width", 200, "display.max_colwidth", 40)
landscape.set_index("Paradigm")

**Architect lens:** moving right in that table you gain *flexibility* and lose *predictability, testability and cost control*.
The professional reflex is: **use the leftmost paradigm that solves the problem.** Day 2 onward you will build with GenAI; today you build the lifecycle discipline that every one of these rows needs.

### A mini "agentic loop" (no LLM required)
An agent is just a **loop**: *decide next step -> call a tool -> observe -> repeat until done*. In production the "decide" step is an LLM. Here we use a deterministic stand-in so you can see the **control structure** and, more importantly, where the **guardrails** live.

In [ ]:
# --- Tools the agent is allowed to call (an explicit allow-list = a security boundary) ---
CRM = {"C-1001": {"name": "A. Rao", "monthly_spend": 82.0, "tenure": 5, "churn_score": 0.71}}

def tool_lookup_customer(cid):   return CRM.get(cid, {})
def tool_send_offer(cid, value): return {"status": "offer_sent", "customer": cid, "value": value}

TOOLS = {"lookup_customer": tool_lookup_customer, "send_offer": tool_send_offer}

def stand_in_planner(goal, history):
    """Pretend-LLM: picks the next action from what it has observed so far."""
    seen = {step["tool"]: step["result"] for step in history}
    if "lookup_customer" not in seen:
        return {"tool": "lookup_customer", "args": {"cid": goal["customer_id"]}}
    cust = seen["lookup_customer"]
    if cust.get("churn_score", 0) > 0.5 and "send_offer" not in seen:
        return {"tool": "send_offer", "args": {"cid": goal["customer_id"], "value": 120}}
    return {"tool": "finish", "args": {}}

def run_agent(goal, max_steps=5, offer_budget=100):
    history = []
    for step in range(max_steps):                         # GUARDRAIL 1: bounded loop
        action = stand_in_planner(goal, history)
        if action["tool"] == "finish":
            break
        if action["tool"] not in TOOLS:                   # GUARDRAIL 2: allow-list
            raise ValueError("unknown tool")
        if action["tool"] == "send_offer" and action["args"]["value"] > offer_budget:
            history.append({"tool": "send_offer", "result": "BLOCKED: exceeds budget -> human approval needed"})
            print(f"  step {step}: send_offer BLOCKED by guardrail (value {action['args']['value']} > budget {offer_budget})")
            continue                                      # GUARDRAIL 3: policy check on side effects
        result = TOOLS[action["tool"]](**action["args"])
        history.append({"tool": action["tool"], "result": result})
        print(f"  step {step}: {action['tool']}({action['args']}) -> {result}")
    return history

print("Run 1 - default budget of 100:")
_ = run_agent({"customer_id": "C-1001"})

**Your turn (2 min):** The agent tried to send a 120 offer and the guardrail blocked it. Re-run the agent so the offer is **allowed** - but think about *who* should be allowed to raise `offer_budget` in production (the agent? the operator? a human approver?).

*Hint: `run_agent(..., offer_budget=...)`.*

In [ ]:
# TODO: call run_agent with an offer_budget that lets the 120 offer through.
# Then answer in a comment: who should own that parameter in production?
_ = run_agent({"customer_id": "C-1001"})   # <-- change this line

**Takeaway:** "Agentic" does not mean "magic". It means *autonomy over actions*, so architecture must add **bounded loops, tool allow-lists, policy checks and audit logs**. We will return to this on Day 4.

---
# Part B - Three flavours of ML in 15 minutes
What separates the three families is not the algorithm, it is the **feedback signal**:

| Family | What the system is given | Feedback signal | Example question |
|---|---|---|---|
| Supervised | Inputs **and correct answers** (labels) | "You were right / wrong" per example | Will this customer churn? |
| Unsupervised | Inputs only | None - find structure yourself | What natural customer segments exist? |
| Reinforcement | An environment to act in | A **reward** after actions (often delayed) | Which sequence of actions maximises long-term reward? |

## B.1 Supervised learning - learning from labelled examples
**Why:** Imagine training a new loan officer by showing them 400 past applications *with the outcome*. They learn where the "approve / deny" line sits. That line is the **decision boundary**. We use just two features so you can literally *see* it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Synthetic loan applications: approve if income is high relative to debt, plus human-like noise.
n = 400
income_k = rng.uniform(20, 120, n)            # annual income, thousands
debt_ratio = rng.uniform(0.05, 0.9, n)         # debt-to-income
score = 0.06 * (income_k - 60) - 5.0 * (debt_ratio - 0.4) + rng.normal(0, 1.0, n)
approved = (score > 0).astype(int)
X_loan = np.column_stack([income_k, debt_ratio])

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_loan, approved, test_size=0.25, random_state=SEED, stratify=approved)
loan_model = LogisticRegression().fit(Xl_tr, yl_tr)

def plot_boundary(model, X, y, title, ax=None):
    """Colour the plane by what the model would predict, then overlay the real labelled points."""
    ax = ax or plt.gca()
    gx, gy = np.meshgrid(np.linspace(20, 120, 250), np.linspace(0.05, 0.9, 250))
    zz = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, zz, levels=[-0.5, 0.5, 1.5], colors=["#F6C9BC", "#D9EDB0"], alpha=0.9)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c=CORAL, s=18, edgecolor="white", lw=0.4, label="denied")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c=TEAL, s=18, edgecolor="white", lw=0.4, label="approved")
    ax.set(xlabel="Income (k)", ylabel="Debt-to-income ratio", title=title)
    ax.legend(loc="upper right", fontsize=8)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
plot_boundary(loan_model, Xl_te, yl_te, f"Logistic regression boundary (test acc = {loan_model.score(Xl_te, yl_te):.2f})", ax)
plt.show()
print("Learned weights  [income, debt_ratio]:", loan_model.coef_.round(3), "| intercept:", loan_model.intercept_.round(2))

**Your turn (3 min):** The straight line is *one* hypothesis. Train a `DecisionTreeClassifier(max_depth=...)` on the same data. Try `max_depth=2` and `max_depth=12`, and compare **train vs test accuracy**. Which one is memorising the training set (overfitting)?

In [ ]:
# TODO: fit a DecisionTreeClassifier with max_depth=12 on (Xl_tr, yl_tr) and print train & test accuracy.
tree = None
if tree is not None:
    print("train acc", tree.score(Xl_tr, yl_tr), "test acc", tree.score(Xl_te, yl_te))
else:
    print("Fill in the TODO above.")

**Architect takeaway (supervised):** needs **labelled historical outcomes** - the label *is* the expensive part (who produces it, how late does it arrive, is it biased?). The train-vs-test gap is your first overfitting alarm.

## B.2 Unsupervised learning - finding structure without answers
**Why:** Marketing asks "who are our customers?" - nobody has labelled them. K-Means groups points that are *close together*. The output is a hypothesis for humans to name ("bargain hunters", "high-value regulars"), **not** a truth.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Three hidden customer personas (we know them, the algorithm does not)
personas = [("Occasional", (25, 2), (8, 0.8), 150),
            ("Regular",    (70, 8), (12, 1.5), 200),
            ("Power user", (140, 18), (20, 2.5), 100)]
pts = []
for _, (m_spend, m_visit), (s_spend, s_visit), cnt in personas:
    pts.append(np.column_stack([rng.normal(m_spend, s_spend, cnt), rng.normal(m_visit, s_visit, cnt)]))
X_cust = np.vstack(pts)                            # columns: monthly spend, visits/month
X_cust_s = StandardScaler().fit_transform(X_cust)  # K-Means is distance-based -> put features on one scale

km = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X_cust_s)
print("silhouette (k=3):", round(silhouette_score(X_cust_s, km.labels_), 3), "(1.0 = perfectly separated, 0 = overlapping)")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for k, col in zip(range(3), [TEAL, CORAL, GOLD]):
    m = km.labels_ == k
    ax.scatter(X_cust[m, 0], X_cust[m, 1], s=16, c=col, alpha=0.8, label=f"segment {k} (n={m.sum()})")
centers = StandardScaler().fit(X_cust).inverse_transform(km.cluster_centers_)
ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=220, c=INK, edgecolor="white", label="centroids")
ax.set(xlabel="Monthly spend", ylabel="Visits per month", title="K-Means customer segments (unlabelled data)")
ax.legend(fontsize=8); plt.show()

**Your turn (3 min):** We told K-Means `k=3` because *we* built the data. In real life nobody tells you. Loop over `k = 2..6`, compute the silhouette score for each and pick the best `k`. Does the answer match the three personas we hid?

In [ ]:
# TODO: compute silhouette_score for k in range(2, 7) and store {k: score} in `sil`.
sil = {}
print("silhouette by k:", sil if sil else "Fill in the TODO above.")

**Architect takeaway (unsupervised):** needs **no labels** but needs a **human interpretation step** and a business definition of "useful". There is no accuracy to report - validate by *actionability* (do segments respond differently to offers?).

## B.3 Reinforcement learning - learning by trial, error and reward
**Why:** Think of training a dog. Nobody gives it a labelled dataset of "correct moves"; it tries things and gets a treat (reward) or not. Below, an agent learns to walk through a 5x5 grid from the top-left (S) to the goal (G, +10) while avoiding pits (-10), paying a small -0.1 for every step (so shorter is better).
The algorithm is **tabular Q-learning**: keep a table `Q[state, action]` = "how good is this move from here?" and nudge it after every step using the reward received.

```
Q[s,a] <- Q[s,a] + alpha * ( reward + gamma * max(Q[s',:]) - Q[s,a] )
```

In [ ]:
# --- A tiny grid world, numpy only ---
GRID = 5
START, GOAL, PITS = (0, 0), (4, 4), {(1, 3), (3, 1), (2, 2)}
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # up, down, left, right
ARROWS = ["^", "v", "<", ">"]

def step_env(state, a):
    r, c = state
    dr, dc = ACTIONS[a]
    nr, nc = min(max(r + dr, 0), GRID - 1), min(max(c + dc, 0), GRID - 1)   # walls: stay inside the grid
    nxt = (nr, nc)
    if nxt == GOAL:  return nxt, 10.0, True
    if nxt in PITS:  return nxt, -10.0, True
    return nxt, -0.1, False

def train_q(epsilon=0.2, episodes=600, alpha=0.5, gamma=0.95, seed=SEED, max_steps=60):
    """Epsilon-greedy Q-learning. epsilon = probability of EXPLORING (random move) instead of exploiting."""
    r_ = np.random.default_rng(seed)
    Q = np.zeros((GRID, GRID, 4))
    returns = []
    for ep in range(episodes):
        s, total = START, 0.0
        for _ in range(max_steps):
            a = r_.integers(4) if r_.random() < epsilon else int(np.argmax(Q[s]))   # explore vs exploit
            s2, rew, done = step_env(s, a)
            Q[s][a] += alpha * (rew + gamma * (0 if done else Q[s2].max()) - Q[s][a])   # the learning rule
            total += rew; s = s2
            if done: break
        returns.append(total)
    return Q, np.array(returns)

Q, returns = train_q(epsilon=0.2)
smooth = np.convolve(returns, np.ones(25) / 25, mode="valid")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(returns, color=TEAL, alpha=0.25, label="episode reward")
ax.plot(np.arange(len(smooth)) + 24, smooth, color=TEAL, lw=2.2, label="25-episode moving average")
ax.set(xlabel="Episode", ylabel="Total reward", title="Q-learning: reward improves with experience")
ax.legend(); plt.show()

print(f"Mean reward - first 50 episodes: {returns[:50].mean():6.2f}   last 50 episodes: {returns[-50:].mean():6.2f}")
print("\nLearned greedy policy (G=goal, X=pit; the agent starts top-left):")
for r in range(GRID):
    print("  " + " ".join("G" if (r, c) == GOAL else ("X" if (r, c) in PITS else ARROWS[int(np.argmax(Q[r, c]))]) for c in range(GRID)))

**Your turn (3 min):** RL has a famous dilemma - **explore vs exploit**. `epsilon` is the share of moves that are random. Train with `epsilon` in `[0.0, 0.05, 0.2, 0.5]` and compare the average reward **during the first 100 episodes** and **during the last 50**. Questions: (1) which setting earns the most *while learning*? (2) does a purely greedy agent (`epsilon=0`) always get stuck - or only sometimes?

In [ ]:
# TODO: for each epsilon, call train_q(epsilon=e) and fill results[e] = (mean reward first 100 episodes, mean reward last 50).
results = {}
for e in [0.0, 0.05, 0.2, 0.5]:
    pass
print(results if results else "Fill in the TODO above.")

**Architect takeaway (reinforcement):** needs a **simulator or safe environment** and a carefully designed **reward function** (a badly specified reward gets gamed!). Real-world uses: pricing, bidding, robotics, and - in a different form - RLHF for aligning LLMs. Data comes from *interaction*, so exploring in production has a real cost.

**Summary of Part B:** *Supervised = labelled answers. Unsupervised = structure only. Reinforcement = rewards from interaction.* In the next part we go deep on the family you will meet in 80% of enterprise work: supervised learning on tabular data, run through a proper lifecycle.

---
# Part C - The AI lifecycle on one running case: telecom customer churn

## C.0 Two lifecycles, one picture
* **CRISP-DM** (Cross-Industry Standard Process for Data Mining, 1990s) describes the *thinking* loop of a data-science project: six phases, deliberately circular.
* **MLOps** extends that loop *after* deployment: automate, version, monitor, retrain. In practice **Monitoring is just Business Understanding again**, with fresh evidence.

We will do all of it on one case, in this order:
**Business Understanding -> Data Understanding -> Data Preparation -> Modeling -> Evaluation -> Deployment -> Monitoring -> (back to the start).**

*The scenario:* you are the solution architect for a telecom with ~500,000 subscribers. Leadership says: *"Churn is killing us. Use AI."* Your job is to turn that sentence into a system.

In [ ]:
# Draw the lifecycle as a loop. Inner ring = CRISP-DM phases, outer text = the MLOps extension.
phases = ["Business\nUnderstanding", "Data\nUnderstanding", "Data\nPreparation", "Modeling",
          "Evaluation", "Deployment", "Monitoring"]
cols = [GOLD, TEAL, TEAL, TEAL, CORAL, INK, LIME]
ang = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, len(phases), endpoint=False)
fig, ax = plt.subplots(figsize=(7.5, 6))
for i, (p, a, c) in enumerate(zip(phases, ang, cols)):
    x, y = 2.1 * np.cos(a), 2.1 * np.sin(a)
    ax.add_patch(plt.Circle((x, y), 0.72, fc=c, ec="white", lw=2, zorder=3))
    ax.text(x, y, p, ha="center", va="center", fontsize=8, fontweight="bold",
            color="white" if c in (TEAL, INK, CORAL) else INK, zorder=4)
    a2 = ang[(i + 1) % len(phases)]
    ax.annotate("", xy=(2.1 * np.cos(a2) * 0.93, 2.1 * np.sin(a2) * 0.93), xytext=(x * 0.93, y * 0.93),
                arrowprops=dict(arrowstyle="->", color=INK, lw=1.2, connectionstyle="arc3,rad=-0.25"), zorder=2)
ax.text(0, 0, "AI lifecycle\n(CRISP-DM + MLOps)\n\nDrift or new\nbusiness need ->\nloop again", ha="center", va="center", fontsize=9)
ax.set_xlim(-3.4, 3.4); ax.set_ylim(-3.2, 3.2); ax.set_aspect("equal"); ax.axis("off"); ax.grid(False)
ax.set_title("The lifecycle is a loop, not a line")
plt.show()

---
# PHASE 1 of 7 - BUSINESS UNDERSTANDING
> *Output artefact: a one-page problem statement - decision, KPI, cost matrix, success criteria, "is ML needed?" verdict.*

**Why this comes first:** the most expensive failure in AI is a technically excellent model that answers a question nobody can act on. Before touching data we pin down four things:

1. **The decision.** Not "predict churn" but *"which customers get a retention offer this month?"*.
2. **The KPI.** Net retention cost (lost margin + offer spend), not accuracy or AUC. Data science metrics are *means*, not ends.
3. **The asymmetry of errors** - written as a **cost matrix**. A missed churner (false negative) and a wasted offer (false positive) hurt very differently.
4. **Success criteria agreed up-front** - what has to be true for the business to say "worth it", and for us to say "stop".

### Our cost matrix
* A customer who churns costs us their remaining margin: **500** (currency units).
* A retention offer costs **60** whether or not the customer would have left.
* An offer saves the customer **50%** of the time (from past campaigns).

In [ ]:
CLV_LOST   = 500.0     # margin lost if a churner is NOT retained
OFFER_COST = 60.0      # cost of making one retention offer
SAVE_RATE  = 0.50      # probability an offer to a would-be churner actually retains them

# Cost of each cell of the confusion matrix (per customer):
COST_TN = 0.0                                   # not flagged, did not churn -> nothing happens
COST_FP = OFFER_COST                            # flagged, would have stayed -> wasted offer
COST_FN = CLV_LOST                              # not flagged, churned -> full loss
COST_TP = OFFER_COST + (1 - SAVE_RATE) * CLV_LOST   # flagged, churner -> pay offer, still lose half the time

cost_matrix = pd.DataFrame([[COST_TN, COST_FP], [COST_FN, COST_TP]],
                           index=["Actually stays", "Actually churns"], columns=["Not flagged", "Flagged (offer)"])
print("Cost matrix (currency per customer):")
print(cost_matrix.round(0).astype(int).to_string())

# Break-even probability: flag a customer when expected cost of flagging < expected cost of ignoring.
#   p*COST_TP + (1-p)*COST_FP  <  p*COST_FN + (1-p)*COST_TN     =>   p > (FP-TN) / ((FP-TN) + (FN-TP))
THEORETICAL_THRESHOLD = (COST_FP - COST_TN) / ((COST_FP - COST_TN) + (COST_FN - COST_TP))
print(f"\nBreak-even churn probability for making an offer: {THEORETICAL_THRESHOLD:.2f}  (NOT 0.50!)")

**Why that last number matters:** the textbook default threshold of 0.5 assumes both errors cost the same. Here a miss is far more expensive than a wasted offer, so we should act on customers who are only **~24% likely** to churn. We will verify this empirically in the Evaluation phase.

**Your turn (3 min):** Before any ML, what do the two "policies" that need **no model at all** cost per 1,000 customers? Assume the true churn rate is 22%.
* Policy A - *do nothing*
* Policy B - *send an offer to everyone*

These are the **business baselines** any model must beat.

In [ ]:
churn_rate_assumed = 0.22
n_cust = 1000
# TODO: compute the expected cost of each policy for 1,000 customers using COST_* constants above.
cost_do_nothing = None
cost_offer_all  = None
print("Do nothing :", cost_do_nothing)
print("Offer all  :", cost_offer_all)

### "Is ML even needed?" check
Ask honestly: *can a rule do 80% of the job?* A common-sense rule ("flag month-to-month customers with under a year of tenure") is cheap, explainable and instantly deployable. So **the rule becomes our benchmark**: ML must beat it by enough to justify extra cost, complexity and governance. We will quantify this in currency.

**Success criteria (agreed with the business, written down now):**
* ML solution lowers total retention cost vs. the best rule-based policy by **>= 10%** on held-out data.
* Every score must be explainable at a customer-service desk (top drivers).
* Alert & retrain if monitored AUC falls > 0.05 below launch value.

---
# PHASE 2 of 7 - DATA UNDERSTANDING
> *Output artefact: data quality report, EDA findings, list of risks (leakage, bias, gaps), data-access agreements.*

**Why:** you cannot model what you do not understand. In a real project this is where you meet the data owners and discover that "tenure" means three different things in three systems. Here we generate a realistic synthetic customer table so everyone has identical data.

*Notes for the architect:* the generator below is the "source system". It deliberately has: missing values, messy category spellings, and one **sneaky leaky column**. Do not peek at the code in detail - treat it as a database you were handed.

In [ ]:
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

CONTRACTS = ["Month-to-month", "One year", "Two year"]
PAYMENTS  = ["Electronic check", "Credit card", "Bank transfer", "Mailed check"]
NUM_COLS  = ["tenure_months", "monthly_charges", "support_tickets_90d", "usage_trend_pct"]
CAT_COLS  = ["contract_type", "payment_method"]

def churn_logit(d, intercept, price_sens=0.02, loyalty=0.6, extra=0.0, competitor=0.0):
    """The hidden 'truth' of why customers leave. Drift in production = these coefficients changing."""
    m2m = (d["contract_type"] == "Month-to-month").to_numpy()
    contract_eff = np.select([m2m, d["contract_type"].eq("One year").to_numpy()], [1.1, -0.2], default=-1.2)
    return (intercept + extra
            - loyalty * np.log(d["tenure_months"])
            + price_sens * (d["monthly_charges"] - 65)
            + contract_eff
            + 0.28 * d["support_tickets_90d"]
            + 0.45 * (d["payment_method"] == "Electronic check")
            + 0.03 * np.maximum(-d["usage_trend_pct"], 0)          # shrinking usage -> leaving
            + 0.7 * (m2m & (d["monthly_charges"] > 85).to_numpy())   # expensive + no commitment: interaction effect
            + competitor * (d["tenure_months"] > 36).to_numpy())          # a rival targets LOYAL customers (concept drift)

def make_customers(n, rg, m2m_share=0.55, charge_shift=0.0, intercept=0.0, price_sens=0.02,
                   loyalty=0.6, extra=0.0, competitor=0.0, start_id=0, messy=True):
    """Generate n customers + churn outcome. Parameters let us create DRIFTED data later."""
    contract = rg.choice(CONTRACTS, n, p=[m2m_share, (1 - m2m_share) * 0.55, (1 - m2m_share) * 0.45])
    bonus = pd.Series(contract).map({"Month-to-month": 0, "One year": 6, "Two year": 12}).to_numpy()
    d = pd.DataFrame({
        "customer_id": [f"C{start_id + i:06d}" for i in range(n)],
        "tenure_months": np.clip(np.round(rg.gamma(2.0, 11, n) + bonus), 1, 72),
        "monthly_charges": np.round(np.clip(rg.normal(65, 22, n) + charge_shift, 18, 140), 2),
        "support_tickets_90d": rg.poisson(1.1, n),
        "contract_type": contract,
        "payment_method": rg.choice(PAYMENTS, n, p=[0.35, 0.28, 0.25, 0.12]),
        "usage_trend_pct": np.round(rg.normal(-2, 20, n), 1),
    })
    p = sigmoid(churn_logit(d, intercept, price_sens, loyalty, extra, competitor) + rg.normal(0, 0.4, n))   # + noise
    d["churned"] = (rg.random(n) < p).astype(int)

    # LEAKY column: logged at cancellation time -> only exists BECAUSE the customer is leaving.
    d["cancellation_request_logged"] = np.where(d["churned"] == 1, rg.random(n) < 0.92, rg.random(n) < 0.01).astype(int)

    if messy:   # real data is never clean: missing values + inconsistent spellings
        for col, frac in [("monthly_charges", 0.03), ("usage_trend_pct", 0.06), ("tenure_months", 0.005)]:
            d.loc[rg.random(n) < frac, col] = np.nan
        variants = {"Month-to-month": ["month-to-month", "Month to month", "MONTH-TO-MONTH "],
                    "One year": ["one year", "One-Year", "ONE YEAR"], "Two year": ["two year", "Two-year", "TWO YEAR "],
                    "Electronic check": ["electronic_check", "E-check", "electronic check "],
                    "Credit card": ["credit card", "Credit Card "], "Bank transfer": ["bank_transfer", "Bank Transfer"],
                    "Mailed check": ["mailed_check", "mailed check"]}
        for col in CAT_COLS:
            mess = rg.random(n) < 0.12
            d.loc[mess, col] = [rg.choice(variants[v]) for v in d.loc[mess, col]]
    return d

# Calibrate the intercept so that the base population churns at ~22% (bisection on a big clean sample).
def _base_rate(b):
    t = make_customers(40000, np.random.default_rng(1), intercept=b, messy=False)
    return t["churned"].mean()
lo, hi = -4.0, 4.0
for _ in range(25):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if _base_rate(mid) < 0.22 else (lo, mid)
BASE_INTERCEPT = round((lo + hi) / 2, 3)

df = make_customers(8500, np.random.default_rng(SEED), intercept=BASE_INTERCEPT)
print(f"Generated {len(df):,} customers | calibrated intercept = {BASE_INTERCEPT}")
df.head()

### First look: shape, types, missingness, imbalance
**Why:** the first five minutes with any dataset should answer: *how big, what types, how dirty, how rare is the thing I'm predicting?*

In [ ]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes.to_string())

miss = df.isna().mean().mul(100).round(2)
print("\nMissing values (%):\n", miss[miss > 0].to_string())

print("\nClass balance (churned):")
print(df["churned"].value_counts(normalize=True).rename({0: "stayed", 1: "churned"}).mul(100).round(1).to_string())

print("\nMessy categories - contract_type as stored in the source system:")
print(df["contract_type"].value_counts().to_string())

**What you should have noticed:**
* `contract_type` has **12 spellings** of 3 real values (`"MONTH-TO-MONTH "` with a trailing space!). A naive one-hot encoder would treat each as a separate category.
* `usage_trend_pct` is ~6% missing; `monthly_charges` ~3%. *Why* they are missing matters (a billing-system outage vs. customers with no data plan) - ask the data owner.
* ~22% churn: **imbalanced** but workable. Accuracy would be misleading (always saying "stays" scores ~78%!) - another reason we chose a cost-based KPI.

We fix the spelling issue with one reusable function, which will later become **part of the deployed pipeline** (so serving code cleans data exactly like training code).

In [ ]:
def clean_categoricals(X: pd.DataFrame) -> pd.DataFrame:
    """Normalise messy category strings: trim, lowercase, treat - and _ as spaces, fix known aliases."""
    X = X.copy()
    aliases = {"e check": "electronic check"}
    for col in CAT_COLS:
        if col in X.columns:
            s = X[col].astype(object).where(X[col].notna(), np.nan)
            s = s.map(lambda v: v if pd.isna(v) else aliases.get(" ".join(str(v).strip().lower().replace("-", " ").replace("_", " ").split()),
                                                              " ".join(str(v).strip().lower().replace("-", " ").replace("_", " ").split())))
            X[col] = s
    return X

eda = clean_categoricals(df)
print("After cleaning:")
print(eda["contract_type"].value_counts().to_string())
print(eda["payment_method"].value_counts().to_string())

### Exploratory plots
**Why:** EDA is hypothesis generation. Each plot should leave you with a sentence you could say to the business: *"Month-to-month customers churn at 3x the rate of two-year customers."*

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

# (a) churn by contract type
g = eda.groupby("contract_type")["churned"].mean().mul(100).sort_values()
axes[0, 0].barh(g.index, g.values, color=[LIME, GOLD, CORAL])
axes[0, 0].set(title="Churn rate by contract type", xlabel="% churned")
for i, v in enumerate(g.values): axes[0, 0].text(v + 0.5, i, f"{v:.0f}%", va="center")

# (b) tenure distributions
for lab, col in [(0, TEAL), (1, CORAL)]:
    axes[0, 1].hist(eda.loc[eda.churned == lab, "tenure_months"].dropna(), bins=30, alpha=0.65, color=col, density=True,
                    label="churned" if lab else "stayed")
axes[0, 1].set(title="Tenure: churners leave early", xlabel="tenure (months)", ylabel="density"); axes[0, 1].legend()

# (c) churn vs monthly charge decile
dec = pd.qcut(eda["monthly_charges"], 10)
cg = eda.groupby(dec, observed=True)["churned"].mean().mul(100)
axes[1, 0].plot(range(1, 11), cg.values, marker="o", color=CORAL, lw=2)
axes[1, 0].set(title="Churn rises with price (by charge decile)", xlabel="monthly charges decile (1=cheapest)", ylabel="% churned")

# (d) churn vs support tickets
tk = eda.assign(t=eda["support_tickets_90d"].clip(upper=6)).groupby("t")["churned"].agg(["mean", "size"])
axes[1, 1].bar(tk.index, tk["mean"] * 100, color=TEAL)
axes[1, 1].set(title="More support tickets, more churn", xlabel="tickets in last 90 days (6 = 6+)", ylabel="% churned")
plt.tight_layout(); plt.show()

### Leakage hunt - the silent project killer
**What is leakage?** A feature that contains information **from the future** (or from the outcome itself) which will *not* be available at the moment you need to make the prediction. The model looks brilliant in the notebook and **collapses in production**, because in production that column is empty or arrives too late.

**Analogy:** predicting whether a student passes an exam, using a feature "received the certificate". Perfect accuracy, zero value.

**A practical detector:** score every feature *on its own* against the label. Anything near-perfect deserves suspicion and a conversation with the data owner: *"At what point in the business process is this value written?"*

In [ ]:
from sklearn.metrics import roc_auc_score

num_all = NUM_COLS + ["cancellation_request_logged"]
uni_auc = {}
for c in num_all:
    v = eda[c].fillna(eda[c].median())
    a = roc_auc_score(eda["churned"], v)
    uni_auc[c] = max(a, 1 - a)               # direction-agnostic AUC (0.5 = useless, 1.0 = perfect)
uni_auc = pd.Series(uni_auc).sort_values(ascending=False)
print("Single-feature AUC against churn:")
print(uni_auc.round(3).to_string())

**Your turn (2 min):** Which column(s) are suspicious? Write a rule that flags any feature with single-feature AUC above a threshold you choose, and store the names in `suspects`. Then explain *in one sentence* why it would fail in production.

In [ ]:
# TODO: choose a threshold and build the list of suspicious column names
suspects = []
print("Suspect columns:", suspects)

**Checkpoint:** here is the answer so everyone is aligned. We drop the leaky column (and the ID, which carries no signal and would invite memorisation). To feel the damage, we first compare model quality **with and without** the leak.

In [ ]:
LEAKY = ["cancellation_request_logged"]
ID_COLS = ["customer_id"]

from sklearn.ensemble import HistGradientBoostingClassifier
tmp = eda[NUM_COLS + LEAKY].fillna(eda[NUM_COLS + LEAKY].median())
Xa, Xb, ya, yb = train_test_split(tmp, eda["churned"], test_size=0.3, random_state=SEED, stratify=eda["churned"])
for name, cols in [("WITH leaky column   ", NUM_COLS + LEAKY), ("WITHOUT leaky column", NUM_COLS)]:
    m = HistGradientBoostingClassifier(random_state=SEED).fit(Xa[cols], ya)
    print(f"{name}: test AUC = {roc_auc_score(yb, m.predict_proba(Xb[cols])[:, 1]):.3f}")

print("\nIn production the leaky flag is ~always 0 at scoring time -> the 'great' model would be useless AND dangerous.")
print("Lesson: a too-good-to-be-true metric is a bug until proven otherwise.")

---
# PHASE 3 of 7 - DATA PREPARATION
> *Output artefact: a versioned, reusable preprocessing + feature pipeline and a documented train / validation / test split.*

**Why three splits, not two?**
* **Train** - the model learns from this.
* **Validation** - we *tune and choose* with this (model type, **decision threshold**). Looking at it repeatedly "uses it up".
* **Test** - touched **once**, at the end, to give an honest estimate of production performance. Think of it as the sealed exam paper.

We stratify so every split has the same 22% churn rate.

**Why a `Pipeline`?** Imputation values, scaling statistics, and the list of one-hot categories are all *learned from the training data*. If you do these steps by hand in a notebook and re-implement them in the serving code, the two will eventually disagree - **train/serve skew**, one of the nastiest bugs in ML. Wrapping every step in one scikit-learn `Pipeline` means **one object, fitted once, used identically everywhere**. That object - not the bare model - is what you deploy and version.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import clone

FEATURES = NUM_COLS + CAT_COLS
X_all, y_all = df[FEATURES], df["churned"]          # note: leaky column and ID are NOT in FEATURES

# 60 / 20 / 20 stratified split (test carved first, then validation out of the remainder)
X_tmp, X_test, y_tmp, y_test = train_test_split(X_all, y_all, test_size=0.20, stratify=y_all, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=SEED)
print(f"train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}")
print("churn rate  train/val/test:", [round(s.mean(), 3) for s in (y_train, y_val, y_test)])

def build_preprocessor(scale=True):
    """Everything the model needs to turn a raw customer row into numbers - all steps learned on train only."""
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]   # indicator: 'was missing' can itself be a signal
    if scale: num_steps.append(("scale", StandardScaler()))
    return Pipeline([
        ("clean", FunctionTransformer(clean_categoricals)),                          # fix spellings, same code in serving
        ("columns", ColumnTransformer([
            ("num", Pipeline(num_steps), NUM_COLS),
            ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT_COLS),
        ])),
    ])

prep_demo = build_preprocessor().fit(X_train)
Z = prep_demo.transform(X_train.head(3))
print(f"\nRaw row -> {Z.shape[1]} numeric model inputs. Example (first customer, first 8 values):")
print(Z[0, :8].round(2))

**Architect lens:** notice what we did *not* do - we never computed medians or scaling on the whole dataset before splitting. That would leak test-set statistics into training (a subtle leakage cousin). The pipeline guarantees `fit` only ever sees training rows.

---
# PHASE 4 of 7 - MODELING
> *Output artefact: candidate models, experiment log, chosen champion.*

**Why start with baselines:** a model is only impressive relative to something. We stack candidates in order of complexity:
1. **Dummy** - always predicts the base rate (the "know-nothing" floor, AUC 0.5).
2. **Rule** - `month-to-month AND tenure < 12 months` (what a smart analyst would write in 10 minutes).
3. **Logistic regression** - simple, fast, explainable (coefficients).
4. **Gradient boosting** - captures non-linearities and interactions; more powerful, less transparent.

**Rule of thumb:** *the simplest model that meets the business requirement wins.* Complexity has a run cost (compute, monitoring, explanation, audit).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

# --- 1 & 2: baselines ---
dummy = DummyClassifier(strategy="prior").fit(np.zeros((len(y_train), 1)), y_train)

def rule_flag(X):
    """Business rule: month-to-month customers in their first year."""
    Xc = clean_categoricals(X)
    return ((Xc["contract_type"] == "month to month") & (Xc["tenure_months"] < 12)).astype(int).to_numpy()

# --- 3 & 4: learned models, each bundled WITH its preprocessing as one Pipeline ---
models = {
    "Logistic regression": Pipeline([("prep", build_preprocessor(scale=True)),
                                     ("clf", LogisticRegression(max_iter=1000))]),
    "Gradient boosting":   Pipeline([("prep", build_preprocessor(scale=False)),
                                     ("clf", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=250, random_state=SEED))]),
}
t0 = time.time()
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"fitted {name:<20} in {time.time() - t0:4.1f}s")

# Probabilities on VALIDATION data for everyone (rule gives hard 0/1, dummy gives the base rate)
val_scores = {"Dummy (base rate)": dummy.predict_proba(np.zeros((len(y_val), 1)))[:, 1],
              "Rule: M2M & tenure<12": rule_flag(X_val).astype(float)}
val_scores.update({name: pipe.predict_proba(X_val)[:, 1] for name, pipe in models.items()})

cmp_ = pd.DataFrame({name: {"ROC-AUC": roc_auc_score(y_val, s), "Avg precision (PR-AUC)": average_precision_score(y_val, s)}
                     for name, s in val_scores.items()}).T.round(3)
print("\nValidation-set comparison:")
print(cmp_.to_string())

**Reading the table:** AUC measures *ranking quality* - "if I pick one churner and one stayer at random, how often does the model score the churner higher?" PR-AUC (average precision) focuses on the rare class; the dummy's PR-AUC equals the base rate (~0.22), which is the number to beat.
Notice that AUC alone does not tell us **which model saves the most money** - that needs the cost matrix. That is the next phase.

**Peek inside the simple model (explainability for free):**

In [ ]:
lr = models["Logistic regression"]
feat_names = lr.named_steps["prep"].named_steps["columns"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=feat_names).sort_values()
top = pd.concat([coefs.head(5), coefs.tail(5)])
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh([n.replace("num__", "").replace("cat__", "").replace("contract_type_", "contract=").replace("payment_method_", "pay=") for n in top.index],
        top.values, color=[TEAL if v < 0 else CORAL for v in top.values])
ax.axvline(0, color=INK, lw=0.8)
ax.set(title="Logistic regression: what pushes churn up (coral) or down (teal)", xlabel="coefficient (standardised features)")
plt.tight_layout(); plt.show()

---
# PHASE 5 of 7 - EVALUATION
> *Output artefact: evaluation report (technical + business), chosen threshold, go / no-go decision.*

CRISP-DM's Evaluation is two questions: **(1)** is the model technically sound? **(2)** does it meet the *business* success criteria from Phase 1? The second is where most teams stop too early.

## 5a. Technical view - ROC, precision-recall, calibration

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
palette = {"Logistic regression": TEAL, "Gradient boosting": CORAL, "Rule: M2M & tenure<12": GOLD, "Dummy (base rate)": INK}

for name, s in val_scores.items():
    fpr, tpr, _ = roc_curve(y_val, s)
    axes[0].plot(fpr, tpr, color=palette[name], lw=2, label=f"{name} ({roc_auc_score(y_val, s):.2f})")
    if "Dummy" not in name:
        pr, rc, _ = precision_recall_curve(y_val, s)
        axes[1].plot(rc, pr, color=palette[name], lw=2, label=name)
axes[0].plot([0, 1], [0, 1], "--", color="grey", lw=1)
axes[0].set(title="ROC curve (validation)", xlabel="False positive rate", ylabel="True positive rate"); axes[0].legend(fontsize=7)
axes[1].axhline(y_val.mean(), color="grey", ls="--", lw=1, label="base rate")
axes[1].set(title="Precision-recall curve", xlabel="Recall (churners caught)", ylabel="Precision (offers well spent)"); axes[1].legend(fontsize=7)

# Calibration: when the model says '30%', do ~30% of those customers churn?
for name in models:
    frac_pos, mean_pred = calibration_curve(y_val, val_scores[name], n_bins=10, strategy="quantile")
    axes[2].plot(mean_pred, frac_pos, marker="o", color=palette[name], lw=2, label=name)
axes[2].plot([0, 1], [0, 1], "--", color="grey", lw=1, label="perfectly calibrated")
axes[2].set(title="Calibration plot", xlabel="Mean predicted probability", ylabel="Observed churn rate"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

**Why calibration matters to an architect:** our threshold logic ("flag when p > 0.24") is only valid if a score of 0.24 *really means* a 24% chance. A model can rank well (high AUC) yet be badly calibrated; then probability-based business rules, expected-value dashboards and risk reports quietly lie. Points close to the diagonal = trustworthy probabilities.

## 5b. Business view - choosing the threshold with the cost matrix
**Why:** a model outputs a probability. A *decision* needs a threshold. The threshold is a **business parameter, not a data-science one**: move it up and you make fewer offers (cheaper, but miss more churners); move it down and you catch more churners at the price of wasted offers.
We sweep thresholds on the **validation** set and compute total cost at each. The minimum is our operating point.

In [ ]:
def total_cost(y_true, flagged):
    """Total business cost of a set of decisions, using the Phase-1 cost matrix."""
    y, f = np.asarray(y_true).astype(bool), np.asarray(flagged).astype(bool)
    tp, fp, fn = (y & f).sum(), (~y & f).sum(), (y & ~f).sum()
    return tp * COST_TP + fp * COST_FP + fn * COST_FN

THRESHOLDS = np.linspace(0.01, 0.99, 99)
def cost_curve(y_true, p):
    return np.array([total_cost(y_true, p >= t) for t in THRESHOLDS])

per_1000 = lambda c, n: c / n * 1000
fig, ax = plt.subplots(figsize=(8.5, 4.6))
best = {}
for name in models:
    cc = per_1000(cost_curve(y_val, val_scores[name]), len(y_val))
    i = int(np.argmin(cc)); best[name] = (THRESHOLDS[i], cc[i])
    ax.plot(THRESHOLDS, cc, color=palette[name], lw=2, label=f"{name} (min {cc[i]:,.0f} @ t={THRESHOLDS[i]:.2f})")
    ax.scatter([THRESHOLDS[i]], [cc[i]], color=palette[name], s=60, zorder=5, edgecolor=INK)
ax.axvline(0.5, color="grey", ls=":", label="naive default t=0.50")
ax.axvline(THEORETICAL_THRESHOLD, color=LIME, ls="--", lw=2, label=f"theory: t={THEORETICAL_THRESHOLD:.2f}")
ax.set(title="Expected cost per 1,000 customers vs decision threshold (validation set)",
       xlabel="Flag customer if predicted churn probability >= threshold", ylabel="Cost per 1,000 customers")
ax.legend(fontsize=8); plt.show()

CHAMPION = min(best, key=lambda k: best[k][1])
THRESHOLD = round(float(best[CHAMPION][0]), 2)
print(f"Champion by validation cost: {CHAMPION}  |  chosen threshold = {THRESHOLD}  (theory said {THEORETICAL_THRESHOLD:.2f})")

**Insight to say out loud:** the empirically optimal threshold lands close to the break-even probability we derived on paper - *and nowhere near 0.5*. Using the default 0.5 would have meant leaving money on the table. The cost curve is also fairly **flat near the optimum**, which tells you the system is not fragile to small threshold changes (good news for governance).

## 5c. The honest number - test set, and the uplift in currency
The test set has been sealed until now. We apply the champion + the threshold chosen on *validation* and compare against every policy, including the **rule-based baseline** and the **no-model** options.

In [ ]:
p_test = models[CHAMPION].predict_proba(X_test)[:, 1]
flag_model = p_test >= THRESHOLD
flag_rule  = rule_flag(X_test).astype(bool)

policies = {
    "Do nothing":                np.zeros(len(y_test), bool),
    "Offer to everyone":         np.ones(len(y_test), bool),
    "Rule: M2M & tenure<12":     flag_rule,
    f"ML ({CHAMPION})":          flag_model,
}
res = pd.DataFrame({k: {"customers flagged": int(v.sum()),
                        "churners caught": int((v & (y_test.to_numpy() == 1)).sum()),
                        "cost per 1,000": per_1000(total_cost(y_test, v), len(y_test))} for k, v in policies.items()}).T
res["cost per 1,000"] = res["cost per 1,000"].round(0)
print(res.to_string())

c_rule, c_ml = res.loc["Rule: M2M & tenure<12", "cost per 1,000"], res.loc[f"ML ({CHAMPION})", "cost per 1,000"]
uplift = (c_rule - c_ml) / c_rule * 100
print(f"\nML vs best rule: saves {c_rule - c_ml:,.0f} per 1,000 customers = {uplift:.1f}% lower cost")
print(f"At 500,000 subscribers: ~{(c_rule - c_ml) * 500:,.0f} per campaign cycle  (success criterion was >= 10%: {'MET' if uplift >= 10 else 'NOT MET'})")

**Confusion matrix at the chosen threshold** - the same story as the cost table, in counts:

In [ ]:
cm = confusion_matrix(y_test, flag_model)
fig, ax = plt.subplots(figsize=(4.8, 4))
ax.imshow(cm, cmap=ListedColormap(["#E8EEF0"]), alpha=0.6)
for (i, j), v in np.ndenumerate(cm):
    label = [["TN: left alone,\nstays", "FP: wasted offer"], ["FN: missed churner", "TP: churner reached"]][i][j]
    ax.text(j, i, f"{v}\n{label}", ha="center", va="center", fontsize=9, color=[[TEAL, GOLD], [CORAL, TEAL]][i][j], fontweight="bold")
ax.set_xticks([0, 1], ["Not flagged", "Flagged"]); ax.set_yticks([0, 1], ["Stays", "Churns"])
ax.set_xlim(-0.5, 1.5); ax.set_ylim(1.5, -0.5)
ax.set(title=f"Confusion matrix on TEST (t={THRESHOLD})", xlabel="Model decision", ylabel="Reality"); ax.grid(False)
plt.show()
print(f"Recall (share of churners reached): {cm[1, 1] / cm[1].sum():.0%} | Precision (offers going to real churners): {cm[1, 1] / cm[:, 1].sum():.0%}")

**Go / no-go (phase output):** if the uplift meets the agreed criterion, we proceed. If it did not, a mature team **stops here** - that is also a successful outcome of the lifecycle: it cost days, not months.

**Your turn (4 min) - a different threshold policy.** Suppose the retention team can only handle **a fixed capacity of offers: 15% of customers per month**. Instead of a cost-optimal threshold, flag the *top 15% by score*. What does that policy cost vs. the cost-optimal threshold on the test set?

In [ ]:
# TODO: build `flag_capacity` = boolean array flagging the 15% of test customers with the highest p_test.
flag_capacity = np.zeros(len(y_test), bool)
print("flagged:", int(flag_capacity.sum()), " cost per 1,000:", round(per_1000(total_cost(y_test, flag_capacity), len(y_test))))

---
# PHASE 6 of 7 - DEPLOYMENT
> *Output artefact: versioned model package, scoring service (API contract), model card, rollout plan.*

**Why:** a notebook is not a product. Deployment means a stable **contract** - "send me these fields, I return this JSON in < 100 ms" - plus metadata so that months later anyone can answer: *which model made this decision, trained on what, with what threshold?*

Four steps:
1. **Persist** the fitted pipeline with `joblib` (preprocessing + model together).
2. **Wrap** it in a scoring function with **input validation** (never trust callers).
3. **Define the API contract** (JSON in / JSON out).
4. **Record a model card** (version, data fingerprint, metrics, threshold, owner, intended use).

In [ ]:
import joblib

MODEL_VERSION = "1.0.0"
champion_pipe = models[CHAMPION]

# Test-set metrics for the card (computed once, on sealed data)
test_auc = roc_auc_score(y_test, p_test)
data_fingerprint = hashlib.sha256(pd.util.hash_pandas_object(X_train, index=False).values.tobytes()).hexdigest()[:16]

model_card = {
    "model_name": "telecom-churn-risk",
    "model_version": MODEL_VERSION,
    "algorithm": CHAMPION,
    "trained_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "training_rows": int(len(X_train)),
    "training_data_fingerprint": data_fingerprint,           # lets auditors tie a model to a dataset
    "features": FEATURES,
    "excluded_for_leakage": LEAKY,
    "decision_threshold": THRESHOLD,
    "cost_matrix": {"clv_lost": CLV_LOST, "offer_cost": OFFER_COST, "save_rate": SAVE_RATE},
    "metrics_test": {"roc_auc": round(float(test_auc), 3), "cost_per_1000": float(c_ml), "baseline_rule_cost_per_1000": float(c_rule)},
    "intended_use": "Rank customers for monthly retention offers. NOT for pricing, credit or service-denial decisions.",
    "owner": "Customer Analytics - Retention squad",
    "monitoring": {"psi_alert": 0.25, "auc_drop_alert": 0.05},
    "random_seed": SEED,
    "library_versions": {"sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}

model_path = ARTIFACTS / f"churn_model_v{MODEL_VERSION}.joblib"
card_path = ARTIFACTS / f"churn_model_v{MODEL_VERSION}.card.json"
joblib.dump(champion_pipe, model_path)
card_path.write_text(json.dumps(model_card, indent=2))
print(f"Saved model : {model_path}  ({model_path.stat().st_size / 1024:.0f} KB)")
print(f"Saved card  : {card_path}")
print(json.dumps({k: model_card[k] for k in ["model_name", "model_version", "algorithm", "decision_threshold", "metrics_test"]}, indent=2))

### The scoring function = the service's brain
**Why validation matters:** callers will send strings for numbers, negative tenure, a typo in the contract name. The scoring service must **fail loudly and clearly** on bad input rather than silently return a confident but meaningless score. Notice we also return *what to do* (a `recommended_action`), because downstream systems consume decisions, not raw probabilities.

**Architect lens:** the model is loaded **once** at start-up (not per request) - that is why latency is ~milliseconds. Note that `clean_categoricals` lives inside the pipeline, so spellings like `"MONTH-TO-MONTH "` are handled identically in training and serving.

In [ ]:
SERVING_MODEL = joblib.load(model_path)               # in a real service: loaded once at process start
CARD = json.loads(card_path.read_text())

VALID = {"tenure_months": (0, 120), "monthly_charges": (0, 500), "support_tickets_90d": (0, 100), "usage_trend_pct": (-100, 500)}
VALID_CONTRACTS = {"month to month", "one year", "two year"}
VALID_PAYMENTS = {"electronic check", "credit card", "bank transfer", "mailed check"}

def predict_one(customer: dict) -> dict:
    """Score ONE customer. Input: dict (parsed JSON). Output: dict (to be serialised as JSON). Never raises on bad input."""
    errors = []
    for f in ["tenure_months", "contract_type", "payment_method"]:                   # mandatory fields
        if customer.get(f) is None: errors.append(f"missing required field: {f}")
    for f, (lo_, hi_) in VALID.items():                                              # numeric range checks (None = allowed, imputed)
        v = customer.get(f)
        if v is None: continue
        if isinstance(v, bool) or not isinstance(v, (int, float)): errors.append(f"{f} must be a number, got {type(v).__name__}")
        elif not (lo_ <= v <= hi_): errors.append(f"{f}={v} outside allowed range [{lo_}, {hi_}]")
    row = pd.DataFrame([{f: customer.get(f, np.nan) for f in FEATURES}])
    row = clean_categoricals(row)
    ct = row.loc[0, "contract_type"]
    if pd.notna(ct) and ct not in VALID_CONTRACTS:
        errors.append(f"unknown contract_type: {customer.get('contract_type')!r}")
    if pd.notna(row.loc[0, "payment_method"]) and row.loc[0, "payment_method"] not in VALID_PAYMENTS:
        errors.append(f"unknown payment_method: {customer.get('payment_method')!r}")
    if errors:
        return {"status": "error", "errors": errors, "model_version": CARD["model_version"]}

    for f in NUM_COLS: row[f] = pd.to_numeric(row[f], errors="coerce")               # None -> NaN for the imputer
    p = float(SERVING_MODEL.predict_proba(row)[0, 1])
    flag = p >= CARD["decision_threshold"]
    return {"status": "ok",
            "churn_probability": round(p, 4),
            "risk_band": "high" if p >= 0.5 else ("medium" if flag else "low"),
            "recommended_action": "make_retention_offer" if flag else "no_action",
            "model_version": CARD["model_version"],
            "threshold": CARD["decision_threshold"],
            "scored_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}

# --- JSON in / JSON out: this is the contract a client would see ---
request_json = '{"tenure_months": 5, "monthly_charges": 92.5, "support_tickets_90d": 3, "usage_trend_pct": -30, "contract_type": "MONTH-TO-MONTH ", "payment_method": "electronic_check"}'
print(">>> REQUEST :", request_json)
print("<<< RESPONSE:", json.dumps(predict_one(json.loads(request_json)), indent=2))

safe_request = {"tenure_months": 60, "monthly_charges": 45.0, "support_tickets_90d": 0, "usage_trend_pct": None, "contract_type": "Two year", "payment_method": "Credit card"}
print("\n>>> safe, loyal customer (usage_trend missing -> imputed):")
print("<<<", json.dumps(predict_one(safe_request)))

bad_request = {"tenure_months": -3, "monthly_charges": "ninety", "contract_type": "Lifetime", "payment_method": "Credit card"}
print("\n>>> malformed request:")
print("<<<", json.dumps(predict_one(bad_request), indent=2))

**Round-trip test (a mini "smoke test" every deployment pipeline should run):** the model we reloaded from disk must give *exactly* the same predictions as the in-memory one.

In [ ]:
same = np.allclose(SERVING_MODEL.predict_proba(X_test)[:, 1], champion_pipe.predict_proba(X_test)[:, 1])
print("Reloaded model reproduces in-memory predictions exactly:", same)

# Latency: how fast is a single-row score (the API's hot path)?
t0 = time.perf_counter()
for _ in range(50): predict_one(safe_request)
print(f"Average single-request scoring time: {(time.perf_counter() - t0) / 50 * 1000:.1f} ms per call (in-process, no network)")

### Wrapping it as a web API
You do **not** need FastAPI installed for this lab. The next cell just *prints* the ~15 lines that would turn `predict_one` into a production-style HTTP service - useful to show how little code the "service" is compared with everything around it (auth, versioning, monitoring, autoscaling, CI/CD).

In [ ]:
FASTAPI_SNIPPET = '''
# app.py  -- run with:  uvicorn app:app --port 8080
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Optional
import joblib, json

app = FastAPI(title="Churn Risk Service", version="1.0.0")
MODEL = joblib.load("artifacts/churn_model_v1.0.0.joblib")        # loaded ONCE at start-up

class Customer(BaseModel):                                          # schema = the API contract (auto-documented at /docs)
    tenure_months: float = Field(ge=0, le=120)
    monthly_charges: Optional[float] = Field(default=None, ge=0, le=500)
    support_tickets_90d: int = Field(ge=0)
    usage_trend_pct: Optional[float] = None
    contract_type: str
    payment_method: str

@app.post("/v1/score")
def score(c: Customer):
    return predict_one(c.model_dump())                              # same function we tested in the notebook

@app.get("/healthz")                                                # for the load balancer / Kubernetes probes
def health(): return {"status": "ok", "model_version": "1.0.0"}
'''
print(FASTAPI_SNIPPET)

**Deployment patterns to mention (architect lens):**
* **Batch scoring** (this case!): churn offers are monthly - score all 500k customers overnight; no need for a real-time API. *Choose the cheapest pattern that meets the latency need.*
* **Real-time API**: when a decision happens in a live interaction (call-centre agent screen).
* **Safe rollout:** shadow mode (score but don't act) -> canary (5% of traffic) -> full, with a one-click rollback to the previous model version.

---
# PHASE 7 of 7 - MONITORING (closing the MLOps loop)
> *Output artefact: monitoring dashboard, alert rules, retraining policy, incident runbook.*

**Why:** the day you deploy is the day the model starts to age. The world changes; the model doesn't. Three different things can go wrong, and they need different responses:

| Problem | What changes | Example in our case | How you detect it |
|---|---|---|---|
| **Data drift** (covariate shift) | The *inputs* P(X) | New marketing push brings more month-to-month customers; price increase raises charges | Compare input distributions to training (PSI, KS test) - **no labels needed**, so it is an *early warning* |
| **Concept drift** | The *relationship* P(y given X) | A competitor launches; loyal long-tenure customers now also leave; customers become more price sensitive | Model performance (AUC, cost) falls - **needs labels** |
| **Label delay** | *When* you learn the truth | You only know someone churned 60-90 days after scoring them | Plan for it: use drift metrics now, performance metrics later |

**PSI (Population Stability Index)** compares a feature's distribution now against the training baseline: roughly `< 0.1` stable, `0.1-0.25` watch, `> 0.25` significant shift.

We simulate **six months in production**. Months 1-2 are normal. From month 3 a price rise and a shift toward month-to-month contracts begin (data drift); from month 4 customers become more price-sensitive and a competitor starts poaching long-tenure customers the model considers "safe" (concept drift).

In [ ]:
# Month-by-month world: each month the generator's parameters drift further from the training world.
MONTHS = ["M1", "M2", "M3", "M4", "M5", "M6"]
drift_plan = pd.DataFrame({
    "m2m_share":    [0.55, 0.57, 0.62, 0.67, 0.72, 0.75],     # data drift: contract mix
    "charge_shift": [0, 0, 12, 14, 16, 16],                    # data drift: price increase from M3
    "price_sens":   [0.02, 0.02, 0.02, 0.03, 0.04, 0.045],     # concept drift: customers get more price sensitive
    "competitor":   [0, 0, 0, 1.5, 3.0, 3.5],                  # concept drift: rival lures long-tenure 'safe' customers
    "extra":        [0, 0, 0, -0.7, -1.3, -1.6],               # (keeps overall churn climate plausible)
}, index=MONTHS)
print(drift_plan.to_string())

prod_rng = np.random.default_rng(SEED + 1)
prod = {}
for i, m in enumerate(MONTHS):
    r_ = drift_plan.loc[m]
    prod[m] = make_customers(1500, prod_rng, m2m_share=r_.m2m_share, charge_shift=r_.charge_shift, intercept=BASE_INTERCEPT,
                             price_sens=r_.price_sens, competitor=r_.competitor, extra=r_.extra, start_id=100000 + i * 1500)
print(f"\nSimulated {sum(len(v) for v in prod.values()):,} production customers over {len(MONTHS)} months.")

In [ ]:
# --- Drift metrics ---
def psi_numeric(ref, cur, bins=10):
    ref, cur = pd.Series(ref).dropna(), pd.Series(cur).dropna()
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-4
    c = np.histogram(cur, edges)[0] / len(cur) + 1e-4
    return float(np.sum((c - r) * np.log(c / r)))

def psi_categorical(ref, cur):
    cats = sorted(set(ref.dropna()) | set(cur.dropna()))
    r = ref.value_counts(normalize=True).reindex(cats).fillna(0).to_numpy() + 1e-4
    c = cur.value_counts(normalize=True).reindex(cats).fillna(0).to_numpy() + 1e-4
    return float(np.sum((c - r) * np.log(c / r)))

ref_clean = clean_categoricals(X_train)             # the training baseline, cleaned the same way as production
rows = []
for m in MONTHS:
    cur = prod[m]
    Xm, ym = cur[FEATURES], cur["churned"]
    pm = champion_pipe.predict_proba(Xm)[:, 1]
    curc = clean_categoricals(Xm)
    row = {"month": m}
    for f in NUM_COLS: row[f"psi_{f}"] = psi_numeric(ref_clean[f], curc[f])
    for f in CAT_COLS: row[f"psi_{f}"] = psi_categorical(ref_clean[f], curc[f])
    row.update({"auc": roc_auc_score(ym, pm),
                "cost_per_1000": per_1000(total_cost(ym, pm >= THRESHOLD), len(ym)),
                "cost_do_nothing_per_1000": per_1000(total_cost(ym, np.zeros(len(ym), bool)), len(ym)),
                "actual_churn": ym.mean(), "mean_predicted": pm.mean()})
    rows.append(row)
mon = pd.DataFrame(rows).set_index("month")
mon["saving_pct"] = 1 - mon["cost_per_1000"] / mon["cost_do_nothing_per_1000"]   # KPI robust to a rising churn rate
psi_cols = [c for c in mon.columns if c.startswith("psi_")]
print(mon[psi_cols].round(3).to_string())
print()
print(mon[["auc", "cost_per_1000", "cost_do_nothing_per_1000", "saving_pct", "actual_churn", "mean_predicted"]].round(3).to_string())

### The monitoring dashboard
One screen, four questions: *Are the inputs shifting? Is ranking quality holding? Is the money still being saved? Are the probabilities still honest?* Alert lines turn charts into **operational rules** that page someone.

In [ ]:
LAUNCH_AUC = roc_auc_score(y_test, p_test)
AUC_ALERT = LAUNCH_AUC - 0.05
LAUNCH_SAVING = 1 - c_ml / res.loc["Do nothing", "cost per 1,000"]   # share of cost the model removes vs doing nothing, at launch
SAVING_ALERT = LAUNCH_SAVING * 0.7                        # alert if the model delivers < 70% of its launch value
# (We monitor 'saving vs do-nothing' rather than raw cost: raw cost rises whenever the churn rate rises, even if the model is fine.)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
# (a) PSI per feature
for c, col in zip(psi_cols, [TEAL, CORAL, GOLD, LIME, INK, "#7F8C8D"]):
    axes[0, 0].plot(MONTHS, mon[c], marker="o", lw=2, color=col, label=c.replace("psi_", ""))
axes[0, 0].axhline(0.10, color=GOLD, ls="--", lw=1); axes[0, 0].axhline(0.25, color=CORAL, ls="--", lw=1.5)
axes[0, 0].text(0, 0.265, "ALERT 0.25", color=CORAL, fontsize=8); axes[0, 0].text(0, 0.115, "watch 0.10", color=GOLD, fontsize=8)
axes[0, 0].set(title="Data drift: PSI per feature (no labels needed)", ylabel="PSI"); axes[0, 0].legend(fontsize=7, ncol=2)
# (b) AUC
axes[0, 1].plot(MONTHS, mon["auc"], marker="o", lw=2.5, color=TEAL)
axes[0, 1].axhline(LAUNCH_AUC, color=LIME, lw=1.5, label=f"launch AUC {LAUNCH_AUC:.2f}")
axes[0, 1].axhline(AUC_ALERT, color=CORAL, ls="--", lw=1.5, label=f"ALERT < {AUC_ALERT:.2f}")
axes[0, 1].set(title="Concept drift: ranking quality (needs labels)", ylabel="ROC-AUC"); axes[0, 1].legend(fontsize=8)
# (c) business KPI: value delivered by the model
axes[1, 0].plot(MONTHS, mon["saving_pct"] * 100, marker="o", lw=2.5, color=CORAL)
axes[1, 0].axhline(LAUNCH_SAVING * 100, color=LIME, lw=1.5, label=f"launch saving {LAUNCH_SAVING:.0%}")
axes[1, 0].axhline(SAVING_ALERT * 100, color=GOLD, ls="--", lw=1.5, label=f"ALERT < {SAVING_ALERT:.0%}")
axes[1, 0].set(title="Business KPI: % cost removed vs do-nothing", ylabel="% saving"); axes[1, 0].legend(fontsize=8)
# (d) calibration drift
axes[1, 1].plot(MONTHS, mon["actual_churn"] * 100, marker="o", lw=2.5, color=CORAL, label="actual churn %")
axes[1, 1].plot(MONTHS, mon["mean_predicted"] * 100, marker="o", lw=2.5, color=TEAL, label="mean predicted %")
axes[1, 1].set(title="Calibration drift: predicted vs actual churn", ylabel="%"); axes[1, 1].legend(fontsize=8)
plt.suptitle("Churn model monitoring dashboard (6 simulated months)", fontweight="bold", y=1.0)
plt.tight_layout(); plt.show()

### Alert rules -> retraining decision
**Why codify it:** "we'll retrain when it looks bad" is not a policy. A written decision rule is auditable, testable and can be automated (a scheduler or Airflow DAG that raises a ticket or kicks off a retraining pipeline).

**Important nuance - label delay:** in reality, month-5 labels would not be known until month 7-8. The PSI alerts (label-free) would fire **first**; AUC/cost alerts confirm later. Here we have labels immediately for teaching purposes.

In [ ]:
def retrain_decision(mon_df, psi_alert=0.25, auc_alert=AUC_ALERT, saving_alert=SAVING_ALERT):
    """Return (decision, reasons) for each month using simple, explainable rules."""
    out = []
    for m, r in mon_df.iterrows():
        reasons = []
        drifted = [c.replace("psi_", "") for c in psi_cols if r[c] > psi_alert]
        if drifted:                 reasons.append(f"data drift on {drifted}")
        if r["auc"] < auc_alert:    reasons.append(f"AUC {r['auc']:.3f} < {auc_alert:.3f}")
        if r["saving_pct"] < saving_alert: reasons.append(f"saving {r['saving_pct']:.0%} < {saving_alert:.0%}")
        # Decision logic: label-free drift alone -> investigate; confirmed performance/KPI breach -> retrain
        perf_breach = (r["auc"] < auc_alert) or (r["saving_pct"] < saving_alert)
        decision = "RETRAIN" if perf_breach else ("INVESTIGATE" if drifted else "OK")
        out.append((m, decision, "; ".join(reasons) or "-"))
    return pd.DataFrame(out, columns=["month", "decision", "reasons"]).set_index("month")

decisions = retrain_decision(mon)
print(decisions.to_string())
first_retrain = decisions.index[decisions["decision"] == "RETRAIN"]
print("\nFirst RETRAIN trigger:", first_retrain[0] if len(first_retrain) else "none")

### Closing the loop: retrain on fresh labels and compare
Retraining is not "press the same button again". The relationship changed, so we (1) add recent labelled months to the training data, (2) **re-run model selection** - a more flexible model family may now be justified - and (3) re-pick the threshold on the most recent month.
We train on the original data plus months 1-4, choose the model and threshold on month 5, and compare **old champion vs new challenger** on month 6 (data none of them has seen). This is the champion-challenger pattern: **never replace a production model without proving the new one is better on recent data.**

In [ ]:
fresh_train = pd.concat([pd.concat([X_train, X_val]).assign(churned=pd.concat([y_train, y_val]).values)]
                        + [prod[m][FEATURES + ["churned"]] for m in ["M1", "M2", "M3", "M4"]], ignore_index=True)
m5, m6 = prod["M5"], prod["M6"]

candidates = {name: clone(pipe).fit(fresh_train[FEATURES], fresh_train["churned"]) for name, pipe in models.items()}
cand_eval = {}
for name, cand in candidates.items():
    p5 = cand.predict_proba(m5[FEATURES])[:, 1]
    cc = cost_curve(m5["churned"], p5)
    cand_eval[name] = (THRESHOLDS[int(np.argmin(cc))], per_1000(cc.min(), len(m5)))
new_name = min(cand_eval, key=lambda k: cand_eval[k][1])
challenger, new_thr = candidates[new_name], float(cand_eval[new_name][0])
print("Candidates re-trained on fresh data, judged on M5 (cost per 1,000):", {k: round(v[1]) for k, v in cand_eval.items()})

yh = m6["churned"]
old_p, new_p = champion_pipe.predict_proba(m6[FEATURES])[:, 1], challenger.predict_proba(m6[FEATURES])[:, 1]
cmp2 = pd.DataFrame({
    "model": {"champion v1.0.0": CHAMPION, "challenger v1.1.0": new_name},
    "AUC": {"champion v1.0.0": roc_auc_score(yh, old_p), "challenger v1.1.0": roc_auc_score(yh, new_p)},
    "threshold": {"champion v1.0.0": THRESHOLD, "challenger v1.1.0": new_thr},
    "cost per 1,000": {"champion v1.0.0": per_1000(total_cost(yh, old_p >= THRESHOLD), len(yh)),
                       "challenger v1.1.0": per_1000(total_cost(yh, new_p >= new_thr), len(yh))},
})
print("\nEvaluation on M6 (unseen by both):")
print(cmp2.round(3).to_string())
gain = cmp2.loc["champion v1.0.0", "cost per 1,000"] - cmp2.loc["challenger v1.1.0", "cost per 1,000"]
print(f"\nRetraining recovers {gain:,.0f} per 1,000 customers. Promote challenger via canary rollout + update the model card (version 1.1.0).")

**Takeaway for Part C:** the lifecycle is a loop. Monitoring found a business problem (cost creeping up), which sends us back to *Business Understanding* ("has the economics of an offer changed?"), *Data Understanding* ("what new customer mix is this?") and so on. **MLOps is the automation that makes this loop cheap and safe to run repeatedly.**

---
# Part D - When NOT to use AI
The most valuable sentence a solution architect can say in a workshop is: **"You don't need AI for this."** It saves the client money, avoids risk, and builds your credibility for the cases where AI *is* right.

We run three mini-experiments. Each demonstrates a distinct failure mode:
1. **The problem is deterministic** - a rule is exact; ML can only approximate.
2. **Not enough data** - ML overfits and results are a lottery.
3. **No signal** - the inputs simply don't predict the outcome; no algorithm can fix that.

## D.1 Deterministic problems: a rules function beats ML
**Case A - progressive income tax.** The law *is* the specification. Ask an ML model to learn it from examples and you get something that is approximately right inside the range it saw, unexplainable ("why this tax?"), and wrong outside it.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error

def tax_due(income):
    """Progressive slabs (illustrative): 0% to 10k, 10% to 40k, 20% to 100k, 35% above."""
    slabs = [(10_000, 0.00), (40_000, 0.10), (100_000, 0.20), (float("inf"), 0.35)]
    tax, lower = 0.0, 0.0
    for upper, rate in slabs:
        if income > lower:
            tax += (min(income, upper) - lower) * rate
        lower = upper
    return tax

# ML is trained on incomes the bank has seen: 5k - 120k.
inc_train = rng.uniform(5_000, 120_000, 3000)
tax_train = np.array([tax_due(x) for x in inc_train])
ml_tax = GradientBoostingRegressor(random_state=SEED).fit(inc_train.reshape(-1, 1), tax_train)

# Test in-range AND out-of-range (a CEO bonus year!)
for label, lo_, hi_ in [("in-range  (5k-120k)", 5_000, 120_000), ("out-of-range (150k-300k)", 150_000, 300_000)]:
    inc = rng.uniform(lo_, hi_, 1000)
    truth = np.array([tax_due(x) for x in inc])
    pred = ml_tax.predict(inc.reshape(-1, 1))
    rule = np.array([tax_due(x) for x in inc])
    print(f"{label:<26}  rules MAE = {mean_absolute_error(truth, rule):>9.2f}   ML MAE = {mean_absolute_error(truth, pred):>10.2f}")

xs = np.linspace(0, 300_000, 400)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(xs, [tax_due(x) for x in xs], color=TEAL, lw=2.5, label="Rules (the law)")
ax.plot(xs, ml_tax.predict(xs.reshape(-1, 1)), color=CORAL, lw=2, ls="--", label="ML (gradient boosting)")
ax.axvspan(120_000, 300_000, color=GOLD, alpha=0.15, label="never seen in training")
ax.set(title="ML cannot extrapolate: it flat-lines outside its training range", xlabel="Income", ylabel="Tax due"); ax.legend()
plt.show()

**Case B - a checksum.** Card numbers, IBANs, and many IDs include a check digit computed by an algorithm (Luhn / mod-97). Can ML "learn" which numbers are valid? We try on a Luhn-style check.

In [ ]:
def luhn_valid(digits):
    """Deterministic validity check used by payment cards. 100% accurate, 5 lines, fully explainable."""
    total = 0
    for i, d in enumerate(reversed(digits)):
        d = d * 2 if i % 2 == 1 else d
        total += d - 9 if d > 9 else d
    return total % 10 == 0

N, L = 12000, 12
digits = rng.integers(0, 10, size=(N, L))
labels = np.array([luhn_valid(list(r)) for r in digits]).astype(int)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(digits, labels, test_size=0.3, random_state=SEED, stratify=labels)

rf = RandomForestClassifier(n_estimators=60, random_state=SEED, n_jobs=1).fit(Xd_tr, yd_tr)
pred_rf = rf.predict(Xd_te)
rules_pred = np.array([luhn_valid(list(r)) for r in Xd_te]).astype(int)

print(f"Share of valid numbers (base rate)  : {labels.mean():.1%}")
print(f"Rules function  : accuracy = {accuracy_score(yd_te, rules_pred):.1%}")
print(f"Random forest   : accuracy = {accuracy_score(yd_te, pred_rf):.1%}   <- looks good!?")
print(f"Random forest   : valid numbers it actually caught (recall) = {(pred_rf[yd_te == 1] == 1).mean():.1%}")
print(f"Random forest   : ROC-AUC = {roc_auc_score(yd_te, rf.predict_proba(Xd_te)[:, 1]):.3f}  (0.5 = coin flip)")

**Read this carefully:** the forest's 90% accuracy is a mirage - it just says "invalid" every time, and 90% of numbers *are* invalid. Recall on the valid ones is ~0 and AUC ~0.5. (This is the **accuracy trap** on imbalanced data again.) The check digit depends on a precise arithmetic relationship that tree ensembles cannot discover from samples.

**Architect lens:** rules are *exact, free, explainable, auditable, instant*. If the logic can be written down by a domain expert - tax slabs, eligibility criteria, validation, routing by a lookup table - **write it down**. Use ML where the rules are too numerous, fuzzy, or unknown to hand-write.

**Your turn (2 min):** A colleague proposes ML to decide *"is this applicant eligible for the senior discount?"* where the policy says *"age >= 65 OR (age >= 60 AND is_pensioner)"*. Write the rules function and test it on the 4 applicants below. How many labelled examples would ML need to match it, and what would you tell the colleague?

In [ ]:
applicants = [dict(age=70, is_pensioner=False), dict(age=62, is_pensioner=True),
              dict(age=62, is_pensioner=False), dict(age=40, is_pensioner=True)]

def senior_discount(a):
    # TODO: return True/False according to the policy
    return None

print([senior_discount(a) for a in applicants], "<- expected [True, True, False, False]")

## D.2 Too little data: ML overfits and results are a lottery
**Why:** with 30 rows and 10 features, a flexible model can memorise anything. It scores a perfect 100% on the data it saw and little better than chance on new data. Worse - the answer *changes depending on which rows you happened to hold out*, so a single train/test split can fool you in either direction.

We show the **spread of cross-validation scores** with n=30 versus n=3,000 for the same underlying (weak) signal.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

def make_signal(n, seed):
    r_ = np.random.default_rng(seed)
    X = r_.normal(size=(n, 10))
    logit = 0.9 * X[:, 0] + 0.6 * X[:, 1]                       # a genuine, but modest, signal in just 2 of 10 features
    y = (r_.random(n) < sigmoid(logit)).astype(int)
    return X, y

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, n_, col in zip(axes, [30, 3000], [CORAL, TEAL]):
    X_, y_ = make_signal(n_, SEED)
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEED)
    big = RandomForestClassifier(n_estimators=40, random_state=SEED, n_jobs=1)
    scores = cross_val_score(big, X_, y_, cv=cv, scoring="accuracy")
    train_acc = big.fit(X_, y_).score(X_, y_)
    ax.hist(scores, bins=12, color=col, alpha=0.85)
    ax.axvline(scores.mean(), color=INK, lw=2)
    ax.set(title=f"n={n_}: CV accuracy {scores.mean():.2f} +/- {scores.std():.2f}\n(training accuracy = {train_acc:.2f})", xlabel="accuracy across 20 CV folds")
    print(f"n={n_:>4}: train acc={train_acc:.2f} | CV mean={scores.mean():.2f} | CV std={scores.std():.3f} | worst fold={scores.min():.2f}, best fold={scores.max():.2f}")
axes[0].set_ylabel("count")
plt.tight_layout(); plt.show()

**Reading the plot:** with n=30 the training accuracy is a perfect (and meaningless) 1.00, the honest CV estimate is mediocre, and individual folds range from "terrible" to "brilliant" - that is **variance**, and it means any one number you report is luck. With n=3,000 the distribution is tight and trustworthy.

**Your turn (3 min):** Would a simpler model help? Replace the random forest by `LogisticRegression` on the n=30 data and compare the CV mean and std. What does this tell you about *model complexity vs. data volume*?

In [ ]:
# TODO: cross_val_score a LogisticRegression on the n=30 data (X30, y30) using the same cv; print mean and std.
X30, y30 = make_signal(30, SEED)
cv30 = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEED)
print("Fill in the TODO above.")

## D.3 No signal, no model
**Why:** sometimes the inputs simply do not contain the information. Example: predicting *which of two equally qualified candidates will be a top performer* from their CV text length, birthday month and postcode. Here we create the extreme: **labels are pure coin flips** independent of 20 features. A flexible model will still produce an impressive-looking training score. Only a held-out evaluation tells the truth.

In [ ]:
n_ns = 2000
X_noise = rng.normal(size=(n_ns, 20))
y_noise = rng.integers(0, 2, n_ns)                         # random labels: there is nothing to learn
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(X_noise, y_noise, test_size=0.3, random_state=SEED)

hgb_noise = HistGradientBoostingClassifier(random_state=SEED).fit(Xn_tr, yn_tr)
auc_tr = roc_auc_score(yn_tr, hgb_noise.predict_proba(Xn_tr)[:, 1])
auc_te = roc_auc_score(yn_te, hgb_noise.predict_proba(Xn_te)[:, 1])
print(f"TRAIN AUC = {auc_tr:.3f}   <- 'wow!'")
print(f"TEST  AUC = {auc_te:.3f}   <- the truth: coin flip")

# A 'permutation test' sanity check you can run on ANY real project: shuffle the labels and re-score.
# If your real model's CV score is not clearly above the shuffled-label scores, you don't have signal.
real_cv = cross_val_score(LogisticRegression(max_iter=500), X_noise, y_noise, cv=5, scoring="roc_auc").mean()
shuffled = [cross_val_score(LogisticRegression(max_iter=500), X_noise, rng.permutation(y_noise), cv=5, scoring="roc_auc").mean() for _ in range(15)]
print(f"\nCV AUC with these labels : {real_cv:.3f}")
print(f"CV AUC with shuffled labels: {np.mean(shuffled):.3f} +/- {np.std(shuffled):.3f}   -> indistinguishable = no signal")

**Architect lens:** the most important output of an AI feasibility study is sometimes **"the data does not support this decision."** Report that early. It is far cheaper than a deployed model that nobody can tell is random. Related red flags for "no model": the label is subjective/inconsistent between annotators; the outcome depends on information you do not collect; or the process is genuinely random.

## D.4 The "Should we use AI?" checklist
We now bundle everything into a 5-question screening function an architect can run in a discovery workshop. It is deliberately simple and **explainable** - the scoring is the conversation.

| # | Question | Why it matters |
|---|---|---|
| 1 | Can the logic be written as explicit rules? | If yes -> rules (D.1) |
| 2 | Do we have enough relevant, labelled examples? | If no -> overfit / lottery (D.2) |
| 3 | Is there evidence the inputs predict the outcome? | If no -> no signal (D.3) |
| 4 | Can the business tolerate errors (or is there a human in the loop)? | Governance and risk |
| 5 | Is the value per decision times volume worth the build + run cost? | Business case |

In [ ]:
def should_we_use_ai(uc: dict, verbose=True) -> dict:
    """Score a use case dict. Returns {'score','verdict','reasons'} and prints a recommendation."""
    reasons, score = [], 0
    # Q1 - hard stop: deterministic logic
    if uc["rules_can_express_logic"]:
        reasons.append("Q1: logic is expressible as rules -> use a rules engine (exact, cheap, explainable)")
    else:
        score += 1; reasons.append("Q1: logic too complex/unknown for rules -> ML plausible")
    # Q2 - data volume
    n_ = uc["labelled_examples"]
    if n_ >= 5000:  score += 1; reasons.append(f"Q2: {n_:,} labelled examples -> enough for most tabular ML")
    elif n_ >= 500: score += 0.5; reasons.append(f"Q2: {n_:,} examples -> marginal; use simple models, expect variance")
    else:           reasons.append(f"Q2: only {n_:,} examples -> high overfit risk; collect data or use rules/expert judgement")
    # Q3 - signal
    if uc["signal_evidence"]: score += 1; reasons.append("Q3: baseline experiment shows signal in the inputs")
    else:                     reasons.append("Q3: no evidence inputs predict outcome -> run a cheap feasibility test first")
    # Q4 - error tolerance
    tol, hitl = uc["error_tolerance"], uc["human_in_loop"]
    if tol == "high" or hitl: score += 1; reasons.append(f"Q4: errors tolerable (tolerance={tol}, human_in_loop={hitl})")
    else:                     reasons.append("Q4: low error tolerance and no human review -> unacceptable risk for probabilistic output")
    # Q5 - value
    annual_value = uc["decisions_per_year"] * uc["value_per_decision"]
    if annual_value >= uc["estimated_annual_cost"] * 3: score += 1; reasons.append(f"Q5: value {annual_value:,.0f} >= 3x cost {uc['estimated_annual_cost']:,.0f}")
    else:                                               reasons.append(f"Q5: value {annual_value:,.0f} vs cost {uc['estimated_annual_cost']:,.0f} -> weak business case")

    # Verdict
    if uc["rules_can_express_logic"]:               verdict = "DO NOT USE AI - use rules"
    elif uc["error_tolerance"] == "low" and not uc["human_in_loop"]: verdict = "NOT YET - redesign with human-in-the-loop"
    elif score >= 4:                                verdict = "GOOD CANDIDATE - proceed to pilot"
    elif score >= 2.5:                              verdict = "PILOT WITH GUARDRAILS - fix gaps first"
    else:                                           verdict = "NOT YET - gather data / prove signal"
    if verbose:
        print(f"\n=== {uc['name']} ===\nScore {score}/5  ->  {verdict}")
        for r_ in reasons: print("  -", r_)
    return {"score": score, "verdict": verdict, "reasons": reasons}

use_cases = [
    dict(name="Income-tax slab calculator", rules_can_express_logic=True, labelled_examples=1_000_000, signal_evidence=True,
         error_tolerance="low", human_in_loop=False, decisions_per_year=2_000_000, value_per_decision=1.0, estimated_annual_cost=300_000),
    dict(name="Telecom churn offers (our case)", rules_can_express_logic=False, labelled_examples=8_500, signal_evidence=True,
         error_tolerance="high", human_in_loop=True, decisions_per_year=6_000_000, value_per_decision=2.0, estimated_annual_cost=400_000),
    dict(name="Auto-approve large insurance claims, no review", rules_can_express_logic=False, labelled_examples=300, signal_evidence=False,
         error_tolerance="low", human_in_loop=False, decisions_per_year=20_000, value_per_decision=40, estimated_annual_cost=500_000),
]
results_uc = [should_we_use_ai(u) for u in use_cases]

**Your turn (4 min):** Think of a use case from *your own* client work. Fill in the dictionary honestly (be sceptical!) and run the checklist. Compare your verdict with your neighbour's. Which answer did you find hardest to give with evidence rather than optimism?

In [ ]:
# TODO: replace the placeholder values with your own use case
my_use_case = dict(name="My use case: <describe it>", rules_can_express_logic=False, labelled_examples=0, signal_evidence=False,
                   error_tolerance="medium", human_in_loop=False, decisions_per_year=0, value_per_decision=0, estimated_annual_cost=1)
_ = should_we_use_ai(my_use_case)

**Takeaway for Part D:** AI is a tool with a *price tag in data, compute, risk and governance*. The five questions are your first-meeting filter; the experiments are your proof when someone pushes back.

---
# Part E - Key AI capabilities: a fast tour
Four capability families cover most enterprise AI demand: **NLP** (language), **Computer Vision** (images/video), **Speech** (audio) and **Recommenders** (choices). In each section below you see the *input representation* (how the raw thing becomes numbers), a classic model, and the **architecture concern** that matters in practice. These are intentionally small and classical; Days 2-3 return to modern deep / GenAI versions.

## E.1 NLP - classifying support tickets
**Why:** text must be converted to numbers first. **TF-IDF** scores each word by how often it appears in this ticket (term frequency) relative to how common it is across all tickets (inverse document frequency) - rare, discriminative words count more. Then any classifier can work. This is the pre-LLM workhorse and still a very strong, cheap baseline for routing problems.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict, StratifiedKFold

tickets = {
    "billing": [
        "I was charged twice on my bill this month", "Please refund the extra charge on my invoice",
        "My bill is higher than the price we agreed", "Why is there a late fee on my bill",
        "The invoice shows a charge I never agreed to", "I need a refund for the wrong amount on my invoice",
        "Payment failed but the charge still appears on my bill", "Wrong tax amount charged on my invoice",
        "I want to dispute a charge on my bill", "Please explain the roaming charge on my invoice",
        "My discount was not applied to the bill", "Refund still not received for the double charge",
        "Send me a copy of last month's invoice and bill", "The bill amount does not match my payment",
        "I was overcharged on my invoice", "Please correct the billing amount and refund me"],
    "technical": [
        "My internet connection keeps dropping", "The router is blinking red and the internet is down",
        "Internet speed is very slow every evening", "The wifi signal is weak and keeps disconnecting",
        "No signal on my phone at home", "Router keeps restarting and the connection drops",
        "Calls drop and the signal is bad", "Cannot connect to wifi after the router update",
        "Network outage in my area, no internet since morning", "Phone signal is poor and calls keep dropping",
        "The modem is not working and the connection is slow", "Streaming keeps buffering because the internet is slow",
        "Mobile data is not working, no signal", "My wifi connection fails every night",
        "The router stopped working and there is no internet", "Slow connection and dropped calls on my phone"],
    "account": [
        "I forgot my password and cannot log in to my account", "Please change the email address on my account",
        "How do I add a family member to my plan", "I want to close my account",
        "My login is locked after too many attempts", "Update the name on my account after marriage",
        "Transfer my account to another person", "How can I reset my password for my account",
        "I did not receive the verification code to log in", "Please upgrade my plan to the unlimited plan",
        "Move my account to my new address", "I want to cancel my plan next month",
        "Add a second line to my existing account", "My account username shows the wrong name",
        "Change my plan to a cheaper plan", "I cannot log in to my account to update my profile"],
}
texts = [t for cat in tickets.values() for t in cat]
labels_t = [cat for cat, ts in tickets.items() for _ in ts]
print(f"{len(texts)} tickets, categories: { {k: len(v) for k, v in tickets.items()} }")

ticket_clf = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")),
                       ("clf", LogisticRegression(max_iter=1000, C=20))])
cv_pred = cross_val_predict(ticket_clf, texts, labels_t, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
print(f"5-fold CV accuracy (honest estimate): {accuracy_score(labels_t, cv_pred):.2f}")

ticket_clf.fit(texts, labels_t)
new_tickets = ["my wifi is down again", "I was billed for a plan I cancelled", "need to reset my login", "router keeps restarting and speeds are bad"]
probs_t = ticket_clf.predict_proba(new_tickets)
for txt, pr in zip(new_tickets, probs_t):
    top = np.argsort(pr)[::-1]
    print(f"  '{txt}'\n      -> {ticket_clf.classes_[top[0]]} ({pr[top[0]]:.0%});  runner-up: {ticket_clf.classes_[top[1]]} ({pr[top[1]]:.0%})")

# Which words drive each category? (explainability)
vocab = np.array(ticket_clf.named_steps["tfidf"].get_feature_names_out())
for i, c in enumerate(ticket_clf.classes_):
    print(f"  top words for {c:<9}:", ", ".join(vocab[np.argsort(ticket_clf.named_steps['clf'].coef_[i])[::-1][:6]]))

**Your turn (2 min):** Type a ticket of your own that is *ambiguous* (belongs to two categories) or something outside the three categories (e.g. "what are your opening hours?"). What does the classifier do? What would you add to a production design to handle that - think *confidence threshold + human fallback + "other" class*.

In [ ]:
my_ticket = "TODO: write your own ticket text here"
pr = ticket_clf.predict_proba([my_ticket])[0]
print(dict(zip(ticket_clf.classes_, pr.round(2))))

**NLP architect concerns:** language/dialect coverage, PII in text (redaction before storage), label consistency between agents, and - with LLMs - hallucination, prompt injection and per-token cost. The TF-IDF model runs in microseconds on a CPU; an LLM doing the same job costs ~1000x more per ticket, so *measure whether you need it*.

## E.2 Computer Vision - recognising handwritten digits
**Why:** an image is just a grid of numbers (pixel intensities). Here: 8x8 greyscale digits, 64 numbers per image. Modern CV uses convolutional or transformer networks on millions of pixels, but the pipeline shape is the same: *pixels -> features -> class*. We use a classic linear SVM so it trains in a second, and then do the thing that matters most in practice: **look at the mistakes**.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.svm import SVC

digits_ds = load_digits()
Xg_tr, Xg_te, yg_tr, yg_te, img_tr, img_te = train_test_split(digits_ds.data, digits_ds.target, digits_ds.images,
                                                              test_size=0.3, random_state=SEED, stratify=digits_ds.target)
svm = SVC(kernel="linear").fit(Xg_tr, yg_tr)
pred_g = svm.predict(Xg_te)
print(f"Image shape: {digits_ds.images[0].shape} = {digits_ds.data.shape[1]} pixel features | test accuracy: {accuracy_score(yg_te, pred_g):.3f}")

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, im, lab in zip(axes, digits_ds.images[:6], digits_ds.target[:6]):
    ax.imshow(im, cmap="gray_r"); ax.set_title(f"label {lab}", fontsize=9); ax.axis("off")
plt.suptitle("What the model sees: 8x8 grids of pixel intensities", fontsize=10); plt.show()

In [ ]:
# Error analysis: a grid of misclassified examples (true label vs prediction)
wrong = np.where(pred_g != yg_te)[0]
show = wrong[:12]
fig, axes = plt.subplots(2, 6, figsize=(11, 4.2))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(img_te[i], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"true {yg_te[i]} / pred {pred_g[i]}", color=CORAL, fontsize=9)
for ax in axes.ravel()[len(show):]: ax.axis("off")
plt.suptitle(f"{len(wrong)} of {len(yg_te)} test digits misclassified - several are ambiguous even to humans", fontsize=11); plt.tight_layout(); plt.show()

# Which classes get confused? (error analysis beats a single accuracy number)
cm_g = confusion_matrix(yg_te, pred_g)
np.fill_diagonal(cm_g, 0)
pairs = [(int(i), int(j), int(cm_g[i, j])) for i, j in zip(*np.nonzero(cm_g))]
print("Most common confusions (true -> predicted : count):", sorted(pairs, key=lambda t: -t[2])[:5])

**CV architect concerns:** labelling cost (images need human annotators), **data/domain shift** (a model trained on clean scans fails on phone photos), GPU vs CPU inference cost, edge vs cloud deployment, privacy for faces/people, and *error analysis as a standing process* - the grid above is what a good ML team reviews weekly.

## E.3 Speech - what a speech model actually "hears"
**Why:** a microphone gives a 1-D waveform (air-pressure over time). Raw waveforms are hard to learn from, so most speech systems first convert to a **spectrogram** - a picture of *which frequencies are present at each moment*. After that, speech recognition is largely an *image-like sequence problem*. We synthesise a signal (two rising/falling tones, like syllables) so we need no audio files.

In [ ]:
from scipy import signal

fs = 16000                                           # 16 kHz sampling: the standard for speech
t = np.arange(0, 1.2, 1 / fs)
# "syllable 1": rising pitch 300->900 Hz in first 0.5s ; "syllable 2": steady 500Hz tone + harmonic ; plus background noise
w1 = np.where(t < 0.5, signal.chirp(t, f0=300, f1=900, t1=0.5, method="linear"), 0)
w2 = np.where((t >= 0.65) & (t < 1.15), 0.8 * np.sin(2 * np.pi * 500 * t) + 0.4 * np.sin(2 * np.pi * 1000 * t), 0)
wave = w1 + w2 + 0.15 * rng.normal(size=t.size)

f, tt, Sxx = signal.spectrogram(wave, fs=fs, nperseg=512, noverlap=384)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t, wave, color=TEAL, lw=0.5); axes[0].set(title="1) Raw waveform: amplitude over time (hard to read)", ylabel="amplitude")
axes[1].pcolormesh(tt, f, 10 * np.log10(Sxx + 1e-10), shading="gouraud", cmap="viridis")
axes[1].set(title="2) Spectrogram: frequency content over time (the 'image' a speech model consumes)", xlabel="time (s)", ylabel="frequency (Hz)", ylim=(0, 3000))
axes[1].grid(False); plt.tight_layout(); plt.show()
print(f"1.2 s of audio = {wave.size:,} samples -> spectrogram of {Sxx.shape[0]} frequency bins x {Sxx.shape[1]} time frames")

**The ASR (automatic speech recognition) pipeline, end to end:**

`microphone -> audio capture (16 kHz) -> voice-activity detection -> feature extraction (spectrogram / log-mel) -> acoustic + language model (today: a neural sequence model like Whisper) -> text -> (optional) punctuation, diarisation ("who spoke"), NLP downstream`

**Speech architect concerns:** streaming vs batch (latency budget: ~300 ms for live captions), accents and noise, **consent and recording law**, PII in transcripts (card numbers read aloud), on-device vs cloud, and word-error-rate (WER) measured **on your own audio**, not the vendor's benchmark. Text-to-speech is the mirror pipeline.

## E.4 Recommenders - "customers who bought X also bought..."
**Why:** recommenders model *taste*. The simplest and surprisingly strong approach is **item-item collaborative filtering**: two items are similar if the *same people* buy them. Similarity = cosine of their purchase columns. No product descriptions needed - the crowd's behaviour is the data. We build a synthetic store with three hidden taste groups.

In [ ]:
items = ["Laptop", "Mouse", "Keyboard", "Monitor", "Yoga mat", "Dumbbells", "Water bottle", "Fitness tracker",
         "Cookbook", "Blender", "Coffee grinder", "Chef knife"]
groups = {"tech": [0, 1, 2, 3, 7], "fitness": [4, 5, 6, 7], "kitchen": [8, 9, 10, 11, 6]}
n_users = 300
user_group = rng.choice(list(groups), n_users)
M = np.zeros((n_users, len(items)))
for u, g in enumerate(user_group):
    for it in range(len(items)):
        base = 0.55 if it in groups[g] else 0.04              # people mostly buy within their taste group
        M[u, it] = rng.random() < base
print(f"User-item matrix: {M.shape}, density = {M.mean():.0%} (real matrices are 99%+ empty)")

from sklearn.metrics.pairwise import cosine_similarity
item_sim = pd.DataFrame(cosine_similarity(M.T), index=items, columns=items)

fig, ax = plt.subplots(figsize=(6.8, 5.5))
im = ax.imshow(item_sim.values, cmap="viridis"); ax.grid(False)
ax.set_xticks(range(len(items)), items, rotation=70, ha="right", fontsize=8); ax.set_yticks(range(len(items)), items, fontsize=8)
ax.set_title("Item-item cosine similarity: taste groups emerge"); plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

def also_bought(item, k=3):
    return item_sim[item].drop(item).sort_values(ascending=False).head(k)

for it in ["Laptop", "Yoga mat", "Blender"]:
    print(f"Customers who bought {it:<9} also bought: {', '.join(f'{n} ({s:.2f})' for n, s in also_bought(it).items())}")

**Evaluate against a baseline - always.** Hide one purchased item per user, recommend 3 items from what remains, and check whether the hidden item is in the list ("hit rate @3"). Compare with the **"most popular items"** strategy that needs no personalisation. A recommender that cannot beat popularity is not worth its complexity.

In [ ]:
def hit_rate_at_3(M, rg):
    hits_cf = hits_pop = trials = 0
    popularity = M.sum(0)
    for u in range(M.shape[0]):
        bought = np.flatnonzero(M[u])
        if len(bought) < 2: continue
        hidden = rg.choice(bought)
        history = bought[bought != hidden]
        # CF score: sum of similarity to items in the user's history; exclude already-owned items
        scores = item_sim.values[history].sum(0).copy(); scores[history] = -1
        pop_scores = popularity.astype(float).copy(); pop_scores[history] = -1
        hits_cf += hidden in np.argsort(scores)[::-1][:3]
        hits_pop += hidden in np.argsort(pop_scores)[::-1][:3]
        trials += 1
    return hits_cf / trials, hits_pop / trials, trials

hr_cf, hr_pop, trials = hit_rate_at_3(M, np.random.default_rng(SEED))
print(f"Hit rate @3 over {trials} users:   item-item CF = {hr_cf:.2f}   |   popularity baseline = {hr_pop:.2f}")
print("(Note: similarity here was computed on the same matrix - a quick demo; a real evaluation would split purchases by time.)")

**Your turn (3 min):** Cold start! A **brand-new user** has bought nothing. What does `also_bought`-style CF recommend for them - and what would you do instead? Write a `recommend_for(history_items)` function that falls back to the most popular items when the history is empty.

In [ ]:
def recommend_for(history, k=3):
    # TODO: if history is empty return the k most popular items; otherwise sum similarity scores over the history.
    return []

print("new user       :", recommend_for([]))
print("bought Laptop  :", recommend_for(["Laptop"]))

**Recommender architect concerns:** cold start (new users / new items), feedback loops (you only learn about what you showed), popularity bias, real-time vs nightly batch candidate generation, diversity and fairness, and measuring *business* lift with an A/B test rather than offline hit-rate alone.

## E.5 Capability map - one table to remember

| Capability | Typical input | What becomes numbers | Typical model families | Key architecture concern |
|---|---|---|---|---|
| **NLP** | Text (tickets, emails, documents) | Tokens -> TF-IDF / embeddings | TF-IDF + linear models, transformers, LLMs | PII, language coverage, hallucination, per-token cost |
| **Computer Vision** | Images, video | Pixel grids -> feature maps | CNNs, vision transformers, detectors | Labelling cost, domain shift, GPU/edge inference, privacy |
| **Speech** | Audio streams | Waveform -> spectrogram / log-mel | Neural ASR (e.g. Whisper), TTS models | Latency (streaming), accents/noise, consent, PII in transcripts |
| **Recommenders** | Interaction logs (clicks, purchases) | User x item matrix, embeddings | Item-item CF, matrix factorisation, two-tower nets | Cold start, feedback loops, A/B measurement, freshness |
| *Tabular prediction (our churn case)* | Rows of business features | Encoded columns | Linear models, gradient-boosted trees | Leakage, calibration, drift, explainability |

**Takeaway for Part E:** every capability follows the same lifecycle you just practised. What changes is the *representation* of the input and where the cost and risk concentrate.

---
# Part F - Wrap-up
## F.1 Lifecycle recap: phase -> artefact -> owner -> failure mode
Run the cell to display it as a table you can screenshot for your own project templates.

In [ ]:
recap = pd.DataFrame([
    ("1 Business Understanding", "Problem statement, KPI, cost matrix, success criteria", "Product owner + architect", "Optimising the wrong metric; no one owns the decision"),
    ("2 Data Understanding",     "Data quality report, EDA, leakage review", "Data engineer + data scientist", "Leakage; hidden bias; unusable / inaccessible data"),
    ("3 Data Preparation",       "Versioned pipeline, split policy", "Data scientist + ML engineer", "Train/serve skew; test-set contamination"),
    ("4 Modeling",               "Baselines, candidate models, experiment log", "Data scientist", "Jumping to complex models; no baseline; irreproducible runs"),
    ("5 Evaluation",             "Technical + business report, threshold, go/no-go", "Data scientist + business owner", "Reporting AUC not value; threshold left at 0.5"),
    ("6 Deployment",             "Model package, API contract, model card, rollout plan", "ML engineer + platform + security", "Unvalidated inputs; no rollback; unclear ownership"),
    ("7 Monitoring (MLOps)",     "Dashboards, alert rules, retraining policy", "ML engineer + business owner", "Silent drift; label delay; nobody on call"),
], columns=["Phase", "Key artefact", "Typical owner", "Classic failure mode"]).set_index("Phase")
recap

## F.2 The Architect's checklist
Use this in design reviews. If you can't answer a line, that is the risk you should raise.

**Problem & value**
- [ ] What *decision* does this change, who makes it today, and what is it worth per decision?
- [ ] Is there a rules or process fix that gets 80% of the value? (Part D)
- [ ] Are the errors costed in a **cost matrix**, and is the threshold derived from it?

**Data**
- [ ] Who owns the data, what are the access / privacy constraints, and where is the label *generated* and how late does it arrive?
- [ ] Has someone hunted for **leakage** and asked "is this feature available at prediction time?"

**Model & pipeline**
- [ ] Is there a baseline (rule, dummy) and a clear uplift over it, in currency?
- [ ] Is preprocessing inside the deployable artefact (no train/serve skew)? Is it reproducible (seeds, versions, data fingerprint)?

**Deployment & operations**
- [ ] Batch or real-time - and is that the *cheapest* pattern that meets the need? Latency / throughput / cost budget stated?
- [ ] Input validation, model card, versioning, canary and rollback plan?

**Monitoring & governance**
- [ ] Data drift (PSI), performance (with label delay planned), business KPI, calibration - each with an alert and an owner?
- [ ] Retraining policy and champion-challenger gate documented? Intended-use limits and fairness checks recorded?

## F.3 Homework / stretch tasks
1. **Different threshold policy.** Add a *budget-constrained* policy to the churn case: the retention team can make at most 800 offers per 5,000 customers *and* each offer must have expected value > 0. Compare it with the unconstrained optimum, and report the cost of the constraint.
2. **Add a new feature.** Engineer `charges_per_tenure_month = monthly_charges / tenure_months` (watch for division by zero and missing values) and add it to `NUM_COLS` and the generator-free pipeline. Does AUC improve? Does *cost* improve? Does it change PSI behaviour? Is it *available at prediction time*?
3. **Simulate label delay.** In the monitoring section assume labels arrive three months late: month *m*'s AUC/cost only becomes known in month *m+3*. Re-draw the dashboard to show only what you would know at the end of month 6, and decide *when* you would first have triggered retraining using PSI alone vs. performance alone. How many months of extra loss does label delay cost?
4. *(Optional)* Replace PSI with a two-sample KS test (`scipy.stats.ks_2samp`) per numeric feature and compare alert timing.

## F.4 Key takeaways
1. **AI is a spectrum** (rules -> ML -> DL -> GenAI -> agents). Moving right buys flexibility and costs predictability, money and governance. *Use the leftmost approach that works.*
2. **The feedback signal defines the paradigm:** labels (supervised), structure only (unsupervised), rewards (reinforcement).
3. **Start with the business:** decision, KPI, **cost matrix**. The threshold is a business decision derived from costs, not the default 0.5.
4. **Beat a baseline, in currency.** A model's value is the difference to the best simple alternative, not its AUC.
5. **Leakage and train/serve skew are the silent killers** - the pipeline object (cleaning + imputing + encoding + model) is the deployable artefact.
6. **Deployment is a contract plus metadata:** validated inputs, versioned model, model card, rollback.
7. **Models decay.** Monitor data drift (early, label-free), concept drift (late, needs labels), and the business KPI; codify when to retrain; use champion-challenger.
8. **Know when to say no:** deterministic logic, too little data, and no signal are all valid reasons to not build an ML model.
9. **Every capability (NLP, CV, Speech, Recommenders) follows the same lifecycle;** only the input representation and the dominant architecture concern change.

*Next up (Day 1 afternoon):* turning these ideas into architecture patterns - where the model, data platform, APIs and governance fit together.

In [ ]:
print(f"Lab complete. Total notebook run time: {time.time() - LAB_START:.0f} s")